In [ ]:
"""
=============================================================================
PRE-MODELING 1: persistence of zeros (history rule for the "sin deserción observada" group)
=============================================================================
Question answered: if a sede shows NO dropout in its own history (DESERCION_LAG3 == 0), how often does
it still have a positive dropout rate in the target year? That share is the "false reassurance" rate of
the rule used by the tool to label sedes as "sin deserción observada".

The rule has no parameters (nothing is fitted), so it is evaluated on every eligible row, by target year,
split and segment. Two versions are reported:
  - A: history zero = DESERCION_LAG3 == 0 (includes the imputed zeros, i.e. sedes without Sit_Acad record)
  - B: history zero = DESERCION_LAG3 == 0 and the zero was observed (not imputed)

Input : Data/Processed/panel_final_<panel>.csv (Phase 2 output)
Output: Data/Auditorio_Panel_Final/pre_modelo_persistencia_<panel>.csv and .md
=============================================================================
"""

from pathlib import Path

import numpy as np
import pandas as pd

# =============================================================================
# 0. PATHS
# =============================================================================
BASE_DIR = Path(
    r"C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data"
)
IN_DIR = BASE_DIR / "Processed"
AUDIT_DIR = BASE_DIR / "Auditorio_Panel_Final"
PANELES = ["ofrecen11", "Solo_Primaria"]

# =============================================================================
# 1. GENERAL VARIABLES
# =============================================================================
COL_SEDE = "SEDE_CODIGO"
COL_ANIO = "ANIO_OBJETIVO"
COL_SEGMENTO = "SEGMENTO_MODELO"
COL_TARGET = "TASA_DESERCION_SEDE"
COL_HIST = "DESERCION_LAG3"
COL_HIST_IMPUTADA = "FLAG_DESERCION_LAG3_CERO_IMPUTADO"
USE_COLS = [
    COL_SEDE, COL_ANIO, "SPLIT", COL_SEGMENTO, "ELEGIBLE_MODELO", "TIENE_HIST_DESERCION",
    COL_HIST, COL_HIST_IMPUTADA, COL_TARGET, "ALTO_RIESGO",
]
VERSIONES = {
    "A_incluye_ceros_imputados": lambda d: d[COL_HIST] == 0,
    "B_solo_ceros_observados": lambda d: (d[COL_HIST] == 0) & (d[COL_HIST_IMPUTADA] == 0),
}

# =============================================================================
# 2. FUNCTION DEFINITIONS
# =============================================================================

def summarize(d: pd.DataFrame, in_rule: pd.Series) -> dict:
    """Metrics of the history rule on the rows of d (all of them have history and are eligible)."""
    positive = d[COL_TARGET] > 0
    alto = d["ALTO_RIESGO"].fillna(0).astype(int) == 1
    n, n_rule = len(d), int(in_rule.sum())
    out_rule = ~in_rule
    return {
        "FILAS": n,
        "FILAS_EN_REGLA": n_rule,
        "PCT_EN_REGLA": 100 * n_rule / n if n else np.nan,
        # Share of the rule group that still shows dropout: the false reassurance of the tool
        "PCT_FALSA_TRANQUILIDAD": 100 * float(positive[in_rule].mean()) if n_rule else np.nan,
        # Same share outside the rule, to read how much the history separates the two groups
        "PCT_POSITIVO_FUERA_REGLA": 100 * float(positive[out_rule].mean()) if out_rule.any() else np.nan,
        # Positives and PRIORITARIO sedes that the rule would label as "sin deserción observada"
        "PCT_POSITIVOS_EN_REGLA": 100 * float((positive & in_rule).sum() / positive.sum()) if positive.sum() else np.nan,
        "PCT_PRIORITARIO_EN_REGLA": 100 * float((alto & in_rule).sum() / alto.sum()) if alto.sum() else np.nan,
    }


def evaluate_panel(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    """Evaluate both versions of the rule by target year, split and segment."""
    base = df[(df["ELEGIBLE_MODELO"] == 1) & (df["TIENE_HIST_DESERCION"] == 1)].copy()
    filas = []
    for version, regla in VERSIONES.items():
        in_rule = regla(base)
        cortes = [
            ("TOTAL", lambda d: pd.Series("todas", index=d.index)),
            ("ANIO_OBJETIVO", lambda d: d[COL_ANIO].astype(int).astype(str)),
            ("SPLIT", lambda d: d["SPLIT"]),
            ("SEGMENTO", lambda d: d[COL_SEGMENTO]),
        ]
        for tipo, clave in cortes:
            grupos = clave(base)
            for g, idx in base.groupby(grupos).groups.items():
                fila = {"PANEL": nombre, "VERSION_REGLA": version, "CORTE": tipo, "GRUPO": g}
                fila.update(summarize(base.loc[idx], in_rule.loc[idx]))
                filas.append(fila)
    return pd.DataFrame(filas)


def to_markdown(df: pd.DataFrame) -> str:
    """Minimal markdown table (no extra dependency)."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for r in df.itertuples(index=False):
        lines.append("| " + " | ".join(f"{v:.2f}" if isinstance(v, float) else str(v) for v in r) + " |")
    return "\n".join(lines)


def save_report(tabla: pd.DataFrame, nombre: str) -> None:
    """Write csv and md; a locked OneDrive file must not abort the analysis."""
    try:
        AUDIT_DIR.mkdir(parents=True, exist_ok=True)
        tabla.to_csv(AUDIT_DIR / f"pre_modelo_persistencia_{nombre}.csv", index=False)
        resumen = tabla[tabla["CORTE"].isin(["TOTAL", "ANIO_OBJETIVO", "SEGMENTO"])]
        md = [f"# Persistencia de ceros - panel {nombre}", "", to_markdown(resumen)]
        (AUDIT_DIR / f"pre_modelo_persistencia_{nombre}.md").write_text("\n".join(md), encoding="utf-8")
    except OSError as err:
        print(f"[WARN] Could not write the report of {nombre}: {err}")


# =============================================================================
# 3. EXECUTION
# =============================================================================

def main() -> None:
    for nombre in PANELES:
        path = IN_DIR / f"panel_final_{nombre}.csv"
        df = pd.read_csv(path, usecols=USE_COLS, dtype={COL_SEDE: str}, low_memory=False)
        tabla = evaluate_panel(df, nombre)
        save_report(tabla, nombre)

        print(f"\n=== {nombre}: false reassurance of the history rule (eligible rows with history) ===")
        vista = tabla[(tabla["CORTE"].isin(["TOTAL", "ANIO_OBJETIVO"]))][
            ["VERSION_REGLA", "CORTE", "GRUPO", "FILAS", "PCT_EN_REGLA", "PCT_FALSA_TRANQUILIDAD",
             "PCT_POSITIVO_FUERA_REGLA", "PCT_POSITIVOS_EN_REGLA", "PCT_PRIORITARIO_EN_REGLA"]
        ]
        print(vista.round(1).to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
"""
=============================================================================
PRE-MODELING 2: how to treat the size of the sede in the dropout target
=============================================================================
Compares several ways of predicting dropout while handling the small-sede problem (a rate of 1 / n for
tiny sedes), with the SAME learner and the SAME temporal backtests, so that only the treatment changes.

Methods (all produce a predicted dropout RATE for each sede):
  B0_historia      baseline: the rate of the sede's own history (DESERCION_LAG3)
  B1_tamano        baseline: every sede gets the same rate, so the ranking is driven by size alone
  M1_pos_tasa      only sedes with a positive rate; target = log(rate); weight = students
  M2_pos_K<k>      same, but the rate is shrunk toward the mean with K pseudo-students (K = 10, 50, 200 and the
                   empirical-Bayes K estimated from the training years)
  M3_dos_partes    probability of a positive rate x the rate of M1 (two-part model)
  M4_poisson       all sedes with a record; Poisson loss on the rate with students as weight (equivalent to
                   modeling the number of deserters with the students as exposure)

Priority of a sede = predicted rate x students ^ ALPHA. ALPHA = 1 is the expected number of students at risk,
ALPHA = 0 is the pure rate; the values in between protect the small sedes without ignoring the size.

Backtests: for each test year, train on the earlier target years only. Population: eligible sedes with a
Sit_Acad record (students known); imputed zeros are "sin dato" and stay out.

Input : Data/Processed/panel_final_<panel>.csv (needs the column PESO_ESTUDIANTES)
Output: Data/Auditorio_Panel_Final/pre_modelo_experimento_tamano_<panel>.csv and .md
=============================================================================
"""

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor

# =============================================================================
# 0. PATHS
# =============================================================================
BASE_DIR = Path(
    r"C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data"
)
IN_DIR = BASE_DIR / "Processed"
AUDIT_DIR = BASE_DIR / "Auditorio_Panel_Final"

# =============================================================================
# 1. GENERAL VARIABLES
# =============================================================================
COL_SEDE, COL_ANIO = "SEDE_CODIGO", "ANIO_OBJETIVO"
COL_RATE, COL_PESO, COL_SIZE = "TASA_DESERCION_SEDE", "PESO_ESTUDIANTES", "MATRICULA_TOTAL"
COL_HIST = "DESERCION_LAG3"

PREDICTORES_COMUNES = [
    "DESERCION_LAG3", "MATRICULA_TOTAL", "MATRICULA_PCT_CAMBIO", "PROP_DESPLAZADOS",
    "PROP_LIMITACION", "PROP_ETNIA", "PROP_FLEXIBLE", "FLAG_PDET", "FLAG_ZOMAC",
]
PREDICTORES_ICFES = ["ICFES_PROM_PUNT_GLOBAL", "ICFES_PROM_INSE"]
PANELES = {
    "ofrecen11": PREDICTORES_COMUNES + PREDICTORES_ICFES,
    "Solo_Primaria": PREDICTORES_COMUNES,
}

TEST_YEARS = [2021, 2022, 2023]     # Each one is predicted with the earlier target years only
K_GRID = [10.0, 50.0, 200.0]        # Pseudo-students of the shrinkage; the empirical-Bayes K is added
ALPHAS = [0.0, 0.1, 0.15, 0.2, 0.25, 0.3, 0.5, 0.75, 1.0]  # Priority = rate x students ^ ALPHA
TOP_SHARES = [0.10, 0.20]           # Share of sedes in the priority list
HGB_PARAMS = dict(
    learning_rate=0.05, max_iter=300, max_depth=4, min_samples_leaf=50,
    l2_regularization=1.0, random_state=42,
)
SEED = 42

# =============================================================================
# 2. FUNCTION DEFINITIONS
# =============================================================================

def estimate_k_empirical_bayes(deserters, students) -> float:
    """
    Empirical-Bayes strength of the prior (K = alpha + beta of a Beta-Binomial) by the method of moments
    (Kleinman, 1973). K is the number of pseudo-students that the shrinkage adds: small K means sedes truly
    differ from each other, large K means most of the differences between small sedes are noise.
    """
    d, n = np.asarray(deserters, float), np.asarray(students, float)
    ok = np.isfinite(d) & np.isfinite(n) & (n > 0)
    d, n = d[ok], n[ok]
    big_n = len(d)
    if big_n < 3 or n.sum() - big_n <= 0:
        return np.nan
    m = d.sum() / n.sum()
    if not 0 < m < 1:
        return np.nan
    s = np.sum(n * (d / n - m) ** 2)
    rho = (s / (m * (1 - m)) - (big_n - 1)) / (n.sum() - big_n)
    rho = float(np.clip(rho, 1e-6, 0.99))
    return (1 - rho) / rho


def shrunk_rate(deserters, students, mean_rate, k) -> np.ndarray:
    """Posterior mean of the rate: (deserters + K * mean) / (students + K)."""
    return (np.asarray(deserters, float) + k * mean_rate) / (np.asarray(students, float) + k)


def load_population(nombre: str) -> pd.DataFrame:
    """Eligible rows with a Sit_Acad record (students known); deserters rebuilt from rate x students."""
    df = pd.read_csv(IN_DIR / f"panel_final_{nombre}.csv", dtype={COL_SEDE: str}, low_memory=False)
    if COL_PESO not in df.columns:
        raise KeyError(f"{COL_PESO} is missing: rerun the master panel and Phase 2 with the denominator restored.")
    df = df[(df["ELEGIBLE_MODELO"] == 1) & df[COL_PESO].gt(0) & df[COL_RATE].notna()].copy()
    df["_d"] = np.round(df[COL_RATE] * df[COL_PESO])
    return df


def fit_predict(df_train: pd.DataFrame, df_test: pd.DataFrame, features: list, k_eb: float) -> dict:
    """Train every method on df_train and return {method: predicted rate for df_test}."""
    pos = df_train[df_train[COL_RATE] > 0]
    m_all = df_train["_d"].sum() / df_train[COL_PESO].sum()
    X_pos, X_all, X_test = pos[features], df_train[features], df_test[features]
    out = {}

    out["B0_historia"] = df_test[COL_HIST].fillna(0.0).to_numpy()
    out["B1_tamano"] = np.full(len(df_test), m_all)

    # M1: positives only, log rate, students as weight (conditional rate: it assumes the sede has dropout)
    sev = HistGradientBoostingRegressor(loss="squared_error", **HGB_PARAMS)
    sev.fit(X_pos, np.log(pos[COL_RATE]), sample_weight=pos[COL_PESO])
    sev_pred = np.exp(sev.predict(X_test))
    out["M1_pos_tasa"] = sev_pred

    # M2: positives only, log of the shrunk rate for each K
    for k in K_GRID + ([k_eb] if np.isfinite(k_eb) else []):
        y = shrunk_rate(pos["_d"], pos[COL_PESO], m_all, k)
        mdl = HistGradientBoostingRegressor(loss="squared_error", **HGB_PARAMS)
        mdl.fit(X_pos, np.log(y), sample_weight=pos[COL_PESO])
        etiqueta = "M2_pos_K_EB" if k == k_eb else f"M2_pos_K{k:.0f}"  # Stable label; the K used goes in K_USADO
        out[etiqueta] = np.exp(mdl.predict(X_test))

    # M3: two parts = P(rate > 0) x severity (the first part is only a component of the expected value)
    clf = HistGradientBoostingClassifier(**HGB_PARAMS)
    clf.fit(X_all, (df_train[COL_RATE] > 0).astype(int))
    out["M3_dos_partes"] = clf.predict_proba(X_test)[:, 1] * sev_pred

    # M4: Poisson loss on the rate with students as weight, all sedes with a record
    poi = HistGradientBoostingRegressor(loss="poisson", **HGB_PARAMS)
    poi.fit(X_all, df_train[COL_RATE], sample_weight=df_train[COL_PESO])
    out["M4_poisson"] = poi.predict(X_test)
    return out


def poisson_deviance(d: np.ndarray, mu: np.ndarray) -> float:
    """Mean Poisson deviance of the expected number of deserters (lower is better)."""
    mu = np.clip(mu, 1e-9, None)
    term = np.where(d > 0, d * np.log(np.where(d > 0, d, 1.0) / mu), 0.0)
    return float(np.mean(2 * (term - (d - mu))))


def within_stratum_capture(d: np.ndarray, rate_hat: np.ndarray, quintil: np.ndarray, rng, share: float = 0.10) -> list:
    """
    Risk discrimination NET of size: inside each size quintile, share of the stratum's deserters that fall in
    the top `share` of its sedes ranked by predicted rate. A random list gets `share` (10%).
    """
    caps = []
    for qn in range(5):
        idx = np.where(quintil == qn)[0]
        if len(idx) == 0 or d[idx].sum() == 0:
            caps.append(np.nan)
            continue
        k = int(np.ceil(share * len(idx)))
        jitter = rng.random(len(idx)) * 1e-12 * max(float(np.nanmax(rate_hat[idx])), 1e-12)
        top = idx[np.argsort(-(rate_hat[idx] + jitter))[:k]]
        caps.append(100 * d[top].sum() / d[idx].sum())
    return caps


def evaluate(df_test: pd.DataFrame, rate_hat: np.ndarray, metodo: str, anio: int, panel: str, k_usado=np.nan) -> list:
    """Capture of deserters and fairness to small sedes for each ALPHA and each priority-list size."""
    rng = np.random.default_rng(SEED)
    d = df_test["_d"].to_numpy()
    n_pred = df_test[COL_SIZE].fillna(df_test[COL_SIZE].median()).clip(lower=1).to_numpy()
    quintil = pd.qcut(pd.Series(n_pred).rank(method="first"), 5, labels=False).to_numpy()  # 0 = smallest
    mu = rate_hat * df_test[COL_PESO].to_numpy()
    calib = float(mu.sum() / d.sum()) if d.sum() else np.nan
    dev = poisson_deviance(d, mu)
    intra = within_stratum_capture(d, rate_hat, quintil, rng)

    filas = []
    for alpha in ALPHAS:
        score = rate_hat * n_pred ** alpha
        score = score + rng.random(len(score)) * 1e-12 * max(float(np.nanmax(score)), 1e-12)  # random tie-break
        orden = np.argsort(-score)
        rho = spearmanr(score, d)[0]
        for q in TOP_SHARES:
            k = int(np.ceil(q * len(d)))
            top = np.zeros(len(d), bool)
            top[orden[:k]] = True
            captura = d[top].sum() / d.sum() if d.sum() else np.nan
            fila = {
                "PANEL": panel, "ANIO_TEST": anio, "METODO": metodo, "K_USADO": k_usado, "ALPHA": alpha, "TOP": q,
                "CAPTURA_DESERTORES": 100 * captura,
                "LIFT": captura / q if np.isfinite(captura) else np.nan,
                "PCT_TOP_EN_Q1_PEQUENAS": 100 * float((top & (quintil == 0)).sum() / k),
                "CAPTURA_Q1_PEQUENAS": 100 * float(d[top & (quintil == 0)].sum() / max(d[quintil == 0].sum(), 1)),
                "CAPTURA_Q5_GRANDES": 100 * float(d[top & (quintil == 4)].sum() / max(d[quintil == 4].sum(), 1)),
                "SPEARMAN": rho, "CALIBRACION_ESPERADO_SOBRE_OBSERVADO": calib, "DEVIANZA_POISSON": dev,
                "CAPTURA_DENTRO_ESTRATO_PROM": float(np.nanmean(intra)),
                "CAPTURA_DENTRO_Q1": intra[0], "CAPTURA_DENTRO_Q5": intra[4],
            }
            filas.append(fila)
    return filas


def to_markdown(df: pd.DataFrame) -> str:
    """Minimal markdown table (no extra dependency)."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for r in df.itertuples(index=False):
        lines.append("| " + " | ".join(f"{v:.2f}" if isinstance(v, float) else str(v) for v in r) + " |")
    return "\n".join(lines)


def save_report(tabla: pd.DataFrame, nombre: str) -> None:
    """Write csv and md; a locked OneDrive file must not abort the experiment."""
    try:
        AUDIT_DIR.mkdir(parents=True, exist_ok=True)
        tabla.to_csv(AUDIT_DIR / f"pre_modelo_experimento_tamano_{nombre}.csv", index=False)
        base = tabla[tabla["TOP"] == 0.10]
        resumen = (
            base.groupby(["METODO", "ALPHA"])[
                ["CAPTURA_DESERTORES", "CAPTURA_Q1_PEQUENAS", "PCT_TOP_EN_Q1_PEQUENAS", "SPEARMAN",
                 "CALIBRACION_ESPERADO_SOBRE_OBSERVADO", "DEVIANZA_POISSON"]
            ].mean().reset_index()
        )
        md = [
            f"# Experimento de tratamiento del tamaño - panel {nombre}",
            "\n## Promedio de los años de prueba, lista del 10% de sedes (CAPTURA = % de los desertores del año que quedan en la lista)\n",
            to_markdown(resumen.sort_values(["ALPHA", "CAPTURA_DESERTORES"], ascending=[True, False])),
            "\n## Discriminación de riesgo dentro de cada estrato de tamaño (promedio de los años; una lista al azar captura 10%)\n",
            to_markdown(
                base[base["ALPHA"] == ALPHAS[0]].groupby("METODO")[
                    ["CAPTURA_DENTRO_ESTRATO_PROM", "CAPTURA_DENTRO_Q1", "CAPTURA_DENTRO_Q5"]
                ].mean().reset_index().sort_values("CAPTURA_DENTRO_ESTRATO_PROM", ascending=False)
            ),
            "\n## Detalle por año de prueba\n",
            to_markdown(base.drop(columns=["PANEL", "TOP"])),
        ]
        (AUDIT_DIR / f"pre_modelo_experimento_tamano_{nombre}.md").write_text("\n".join(md), encoding="utf-8")
    except OSError as err:
        print(f"[WARN] Could not write the report of {nombre}: {err}")


def run_panel(nombre: str, features: list) -> pd.DataFrame:
    """Rolling-origin backtests for one panel."""
    df = load_population(nombre)
    filas = []
    for anio in TEST_YEARS:
        train, test = df[df[COL_ANIO] < anio], df[df[COL_ANIO] == anio]
        if train.empty or test.empty:
            print(f"[WARN] {nombre} {anio}: no train or test rows, skipped.")
            continue
        k_eb = estimate_k_empirical_bayes(train["_d"], train[COL_PESO])
        print(f"{nombre} | test {anio} | train rows {len(train):,} | test rows {len(test):,} | K empirical Bayes = {k_eb:.1f}")
        predicciones = fit_predict(train, test, features, k_eb)
        k_por_metodo = {f"M2_pos_K{k:.0f}": k for k in K_GRID}
        k_por_metodo["M2_pos_K_EB"] = k_eb
        for metodo, rate_hat in predicciones.items():
            filas.extend(evaluate(test, rate_hat, metodo, anio, nombre, k_por_metodo.get(metodo, np.nan)))
    return pd.DataFrame(filas)


# =============================================================================
# 3. EXECUTION
# =============================================================================

def main() -> None:
    for nombre, features in PANELES.items():
        tabla = run_panel(nombre, features)
        save_report(tabla, nombre)
        top = tabla[tabla["TOP"] == 0.10]
        print(f"\n=== {nombre}: mean over the test years, list of 10% of the sedes ===")
        print(
            top.groupby(["METODO", "ALPHA"])[["CAPTURA_DESERTORES", "CAPTURA_Q1_PEQUENAS", "CALIBRACION_ESPERADO_SOBRE_OBSERVADO"]]
            .mean().round(2).to_string()
        )


if __name__ == "__main__":
    main()


In [ ]:
"""
=============================================================================
PRE-MODELING 3: does a better use of the sede's own history improve the Poisson model?
=============================================================================
Finding of the size experiment: almost all the risk signal is the sede's own history (DESERCION_LAG3), and
the model barely beats it inside each size stratum (it is weak for the large sedes of Solo_Primaria).
Hypothesis: the raw rate of t-3 is very noisy for small sedes and the model cannot tell how reliable it is.

Feature sets compared with the SAME learner (Poisson, students as weight) and the SAME backtests:
  BASE              current predictors (the history enters only as the raw rate DESERCION_LAG3)
  AJUSTADA          + DESERCION_LAG3_AJUSTADA = (deserters + K * mean) / (students + K), K = empirical Bayes
  CONTEOS           + ESTUDIANTES_LAG3 and DESERTORES_LAG3 (students and deserters of the sede in t-3)
  AJUSTADA+CONTEOS  all of them
Baselines without a model: raw history, shrunk history and size only.

The t-3 counts are rebuilt from the final panel itself by calendar year: the row of the same sede whose
ANIO_OBJETIVO is t-3 carries its students (PESO_ESTUDIANTES) and its rate. Sedes without a Sit_Acad record in
t-3 have no counts (NaN), never zero.

Input : Data/Processed/panel_final_<panel>.csv (needs PESO_ESTUDIANTES)
Output: Data/Auditorio_Panel_Final/pre_modelo_experimento_historia_<panel>.csv and .md
=============================================================================
"""

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor

# =============================================================================
# 0. PATHS
# =============================================================================
BASE_DIR = Path(
    r"C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data"
)
IN_DIR = BASE_DIR / "Processed"
AUDIT_DIR = BASE_DIR / "Auditorio_Panel_Final"

# =============================================================================
# 1. GENERAL VARIABLES
# =============================================================================
COL_SEDE, COL_ANIO = "SEDE_CODIGO", "ANIO_OBJETIVO"
COL_RATE, COL_PESO, COL_SIZE = "TASA_DESERCION_SEDE", "PESO_ESTUDIANTES", "MATRICULA_TOTAL"
COL_HIST = "DESERCION_LAG3"
HIST_LAG = 3

PREDICTORES_COMUNES = [
    "DESERCION_LAG3", "MATRICULA_TOTAL", "MATRICULA_PCT_CAMBIO", "PROP_DESPLAZADOS",
    "PROP_LIMITACION", "PROP_ETNIA", "PROP_FLEXIBLE", "FLAG_PDET", "FLAG_ZOMAC",
]
PREDICTORES_ICFES = ["ICFES_PROM_PUNT_GLOBAL", "ICFES_PROM_INSE"]
PANELES = {
    "ofrecen11": PREDICTORES_COMUNES + PREDICTORES_ICFES,
    "Solo_Primaria": PREDICTORES_COMUNES,
}
CONJUNTOS = {  # name -> extra history features added to the base predictors
    "BASE": [],
    "AJUSTADA": ["DESERCION_LAG3_AJUSTADA"],
    "CONTEOS": ["ESTUDIANTES_LAG3", "DESERTORES_LAG3"],
    "AJUSTADA+CONTEOS": ["DESERCION_LAG3_AJUSTADA", "ESTUDIANTES_LAG3", "DESERTORES_LAG3"],
}

TEST_YEARS = [2021, 2022, 2023]
ALPHAS = [0.0, 0.1, 0.2, 0.3, 1.0]   # Priority = rate x students ^ ALPHA
TOP_SHARE = 0.10
HGB_PARAMS = dict(
    learning_rate=0.05, max_iter=300, max_depth=4, min_samples_leaf=50,
    l2_regularization=1.0, random_state=42,
)
SEED = 42

# =============================================================================
# 2. FUNCTION DEFINITIONS
# =============================================================================

def estimate_k_empirical_bayes(deserters, students) -> float:
    """Empirical-Bayes strength of the prior (K = alpha + beta of a Beta-Binomial), method of moments (Kleinman, 1973)."""
    d, n = np.asarray(deserters, float), np.asarray(students, float)
    ok = np.isfinite(d) & np.isfinite(n) & (n > 0)
    d, n = d[ok], n[ok]
    big_n = len(d)
    if big_n < 3 or n.sum() - big_n <= 0:
        return np.nan
    m = d.sum() / n.sum()
    if not 0 < m < 1:
        return np.nan
    s = np.sum(n * (d / n - m) ** 2)
    rho = float(np.clip((s / (m * (1 - m)) - (big_n - 1)) / (n.sum() - big_n), 1e-6, 0.99))
    return (1 - rho) / rho


def load_panel(nombre: str) -> pd.DataFrame:
    """Final panel plus the students and deserters of the same sede in t-3, joined by calendar year."""
    df = pd.read_csv(IN_DIR / f"panel_final_{nombre}.csv", dtype={COL_SEDE: str}, low_memory=False)
    if COL_PESO not in df.columns:
        raise KeyError(f"{COL_PESO} is missing: rerun the master panel and Phase 2 with the denominator restored.")
    hist = df[[COL_SEDE, COL_ANIO, COL_RATE, COL_PESO]].copy()
    hist[COL_ANIO] = hist[COL_ANIO] + HIST_LAG
    hist = hist.rename(columns={COL_RATE: "_rate_lag3", COL_PESO: "ESTUDIANTES_LAG3"})
    df = df.merge(hist, on=[COL_SEDE, COL_ANIO], how="left", validate="many_to_one")
    df["DESERTORES_LAG3"] = np.round(df["_rate_lag3"] * df["ESTUDIANTES_LAG3"])  # NaN where there is no record

    # The rebuilt history must match the DESERCION_LAG3 column of the panel
    both = df["_rate_lag3"].notna() & df[COL_HIST].notna()
    if both.any() and not np.allclose(df.loc[both, "_rate_lag3"], df.loc[both, COL_HIST], atol=1e-9):
        raise ValueError("The rate rebuilt for t-3 does not match DESERCION_LAG3: check the panel.")
    return df


def population(df: pd.DataFrame) -> pd.DataFrame:
    """Eligible rows with a Sit_Acad record (students known); deserters rebuilt from rate x students."""
    pop = df[(df["ELEGIBLE_MODELO"] == 1) & df[COL_PESO].gt(0) & df[COL_RATE].notna()].copy()
    pop["_d"] = np.round(pop[COL_RATE] * pop[COL_PESO])
    return pop


def add_shrunk_history(train: pd.DataFrame, test: pd.DataFrame, k: float) -> tuple:
    """
    DESERCION_LAG3_AJUSTADA = (deserters_t-3 + K * mean) / (students_t-3 + K). K and the mean come from the TRAIN
    years only (two global constants); a sede without counts in t-3 gets the prior mean.
    """
    m_all = train["_d"].sum() / train[COL_PESO].sum()
    out = []
    for frame in (train, test):
        frame = frame.copy()
        known = frame["ESTUDIANTES_LAG3"].gt(0)
        shrunk = (frame["DESERTORES_LAG3"] + k * m_all) / (frame["ESTUDIANTES_LAG3"] + k)
        frame["DESERCION_LAG3_AJUSTADA"] = np.where(known, shrunk, m_all)
        out.append(frame)
    return out[0], out[1]


def poisson_deviance(d: np.ndarray, mu: np.ndarray) -> float:
    """Mean Poisson deviance of the expected number of deserters (lower is better)."""
    mu = np.clip(mu, 1e-9, None)
    term = np.where(d > 0, d * np.log(np.where(d > 0, d, 1.0) / mu), 0.0)
    return float(np.mean(2 * (term - (d - mu))))


def within_stratum_capture(d, rate_hat, quintil, rng, share=0.10) -> list:
    """Inside each size quintile, % of the stratum's deserters in the top `share` of its sedes by predicted rate (random = 10%)."""
    caps = []
    for qn in range(5):
        idx = np.where(quintil == qn)[0]
        if len(idx) == 0 or d[idx].sum() == 0:
            caps.append(np.nan)
            continue
        k = int(np.ceil(share * len(idx)))
        jitter = rng.random(len(idx)) * 1e-12 * max(float(np.nanmax(rate_hat[idx])), 1e-12)
        top = idx[np.argsort(-(rate_hat[idx] + jitter))[:k]]
        caps.append(100 * d[top].sum() / d[idx].sum())
    return caps


def evaluate(df_test, rate_hat, conjunto, anio, panel) -> list:
    """Capture of deserters, fairness to small sedes and risk discrimination net of size."""
    rng = np.random.default_rng(SEED)
    d = df_test["_d"].to_numpy()
    n_pred = df_test[COL_SIZE].fillna(df_test[COL_SIZE].median()).clip(lower=1).to_numpy()
    quintil = pd.qcut(pd.Series(n_pred).rank(method="first"), 5, labels=False).to_numpy()
    mu = rate_hat * df_test[COL_PESO].to_numpy()
    intra = within_stratum_capture(d, rate_hat, quintil, rng)
    filas = []
    for alpha in ALPHAS:
        score = rate_hat * n_pred ** alpha
        score = score + rng.random(len(score)) * 1e-12 * max(float(np.nanmax(score)), 1e-12)
        top = np.zeros(len(d), bool)
        top[np.argsort(-score)[: int(np.ceil(TOP_SHARE * len(d)))]] = True
        filas.append({
            "PANEL": panel, "ANIO_TEST": anio, "CONJUNTO": conjunto, "ALPHA": alpha,
            "CAPTURA_DESERTORES": 100 * d[top].sum() / d.sum() if d.sum() else np.nan,
            "CAPTURA_Q1_PEQUENAS": 100 * float(d[top & (quintil == 0)].sum() / max(d[quintil == 0].sum(), 1)),
            "SPEARMAN": spearmanr(score, d)[0],
            "CALIBRACION": float(mu.sum() / d.sum()) if d.sum() else np.nan,
            "DEVIANZA_POISSON": poisson_deviance(d, mu),
            "CAPTURA_DENTRO_ESTRATO_PROM": float(np.nanmean(intra)),
            "CAPTURA_DENTRO_Q1": intra[0], "CAPTURA_DENTRO_Q5": intra[4],
        })
    return filas


def to_markdown(df: pd.DataFrame) -> str:
    """Minimal markdown table (no extra dependency)."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for r in df.itertuples(index=False):
        lines.append("| " + " | ".join(f"{v:.2f}" if isinstance(v, float) else str(v) for v in r) + " |")
    return "\n".join(lines)


def save_report(tabla: pd.DataFrame, nombre: str) -> None:
    """Write csv and md; a locked OneDrive file must not abort the experiment."""
    try:
        AUDIT_DIR.mkdir(parents=True, exist_ok=True)
        tabla.to_csv(AUDIT_DIR / f"pre_modelo_experimento_historia_{nombre}.csv", index=False)
        medias = tabla.groupby(["CONJUNTO", "ALPHA"])[
            ["CAPTURA_DESERTORES", "CAPTURA_Q1_PEQUENAS", "SPEARMAN", "CALIBRACION", "DEVIANZA_POISSON"]
        ].mean().reset_index()
        intra = tabla[tabla["ALPHA"] == ALPHAS[0]].groupby("CONJUNTO")[
            ["CAPTURA_DENTRO_ESTRATO_PROM", "CAPTURA_DENTRO_Q1", "CAPTURA_DENTRO_Q5", "DEVIANZA_POISSON"]
        ].mean().reset_index().sort_values("CAPTURA_DENTRO_ESTRATO_PROM", ascending=False)
        # Consistency across years: difference with BASE in the discrimination net of size
        por_anio = tabla[tabla["ALPHA"] == ALPHAS[0]].pivot_table(
            index="CONJUNTO", columns="ANIO_TEST", values="CAPTURA_DENTRO_ESTRATO_PROM"
        )
        ref = "M4_BASE"  # The model labels carry the prefix M4_ (the previous version looked for "BASE" and printed raw values)
        dif = por_anio.sub(por_anio.loc[ref], axis=1).reset_index() if ref in por_anio.index else por_anio.reset_index()
        dif.columns = [str(c) for c in dif.columns]
        md = [
            f"# Experimento de historia - panel {nombre}",
            "\n## Discriminación de riesgo dentro de cada estrato de tamaño (promedio de los años; azar = 10%)\n",
            to_markdown(intra),
            "\n## Diferencia frente a M4_BASE por año de prueba (puntos de captura dentro del estrato; positivo = mejor que BASE)\n",
            to_markdown(dif),
            "\n## Lista del 10% de sedes por conjunto y alpha (promedio de los años)\n",
            to_markdown(medias),
        ]
        (AUDIT_DIR / f"pre_modelo_experimento_historia_{nombre}.md").write_text("\n".join(md), encoding="utf-8")
    except OSError as err:
        print(f"[WARN] Could not write the report of {nombre}: {err}")


def run_panel(nombre: str, base_features: list) -> pd.DataFrame:
    """Rolling-origin backtests for one panel: one Poisson model per feature set plus the baselines."""
    pop = population(load_panel(nombre))
    filas = []
    for anio in TEST_YEARS:
        train, test = pop[pop[COL_ANIO] < anio], pop[pop[COL_ANIO] == anio]
        if train.empty or test.empty:
            print(f"[WARN] {nombre} {anio}: no train or test rows, skipped.")
            continue
        k_eb = estimate_k_empirical_bayes(train["_d"], train[COL_PESO])
        print(f"{nombre} | test {anio} | train rows {len(train):,} | test rows {len(test):,} | K empirical Bayes = {k_eb:.1f}")
        train, test = add_shrunk_history(train, test, k_eb)

        # Baselines without a model
        filas.extend(evaluate(test, test[COL_HIST].fillna(0.0).to_numpy(), "B0_historia_cruda", anio, nombre))
        filas.extend(evaluate(test, test["DESERCION_LAG3_AJUSTADA"].to_numpy(), "B0b_historia_ajustada", anio, nombre))
        filas.extend(evaluate(test, np.full(len(test), train["_d"].sum() / train[COL_PESO].sum()), "B1_tamano", anio, nombre))

        # Poisson with students as weight, one model per feature set
        for conjunto, extra in CONJUNTOS.items():
            features = base_features + extra
            mdl = HistGradientBoostingRegressor(loss="poisson", **HGB_PARAMS)
            mdl.fit(train[features], train[COL_RATE], sample_weight=train[COL_PESO])
            filas.extend(evaluate(test, mdl.predict(test[features]), f"M4_{conjunto}", anio, nombre))
    return pd.DataFrame(filas)


# =============================================================================
# 3. EXECUTION
# =============================================================================

def main() -> None:
    for nombre, features in PANELES.items():
        tabla = run_panel(nombre, features)
        save_report(tabla, nombre)
        print(f"\n=== {nombre}: risk discrimination net of size (mean over test years; random = 10%) ===")
        print(
            tabla[tabla["ALPHA"] == ALPHAS[0]].groupby("CONJUNTO")[
                ["CAPTURA_DENTRO_ESTRATO_PROM", "CAPTURA_DENTRO_Q1", "CAPTURA_DENTRO_Q5", "DEVIANZA_POISSON"]
            ].mean().round(2).sort_values("CAPTURA_DENTRO_ESTRATO_PROM", ascending=False).to_string()
        )


if __name__ == "__main__":
    main()


In [ ]:
"""
=============================================================================
PRE-MODELING 4: the sede's own history as the BASE of the model (offset) vs as one more feature
=============================================================================
Finding of the history experiment: the shrunk history alone (no model) discriminates as well as any variant of
the model inside each size stratum, and a SHAP built on top of it would be dominated by DESERCION_LAG3, which is
not actionable. Idea: let the history set the baseline of each sede and let the model learn only the deviation
from that baseline with the remaining (actionable) predictors. Then SHAP can explain "everything except the lag".

Variants (all Poisson, students as weight, same backtests 2021-2023, same LightGBM settings):
  A    history as FEATURES (raw rate + shrunk rate + t-3 counts), no offset. Strongest feature-based comparator.
  B    OFFSET = Poisson GLM on log(shrunk history) with a FREE slope; the GBM sees only the NON-history
       predictors. This is the SHAP-friendly design (history lives in the offset, outside the explained part).
  B2   same offset, but the GBM also sees the history features (cost of leaving the lag out of the GBM = B vs B2).
  C    B2 + monotone increasing constraint on the two history rates.
  Each one is run with two capacities (number of trees).
Baselines: GLM_solo (the offset model alone), shrunk history without model, and the previous HistGradientBoosting
(M4_BASE_HGB) to check that the earlier numbers are reproduced.

Offset in LightGBM: the GLM linear predictor is passed as init_score at fit time and added back by hand at
prediction time (predict(raw_score=True) does not include it). Students enter as sample weight and the target is
the rate, which is the same as modelling deserters with log(students) as offset.

Uncertainty: paired bootstrap by MUNICIPIO (the same resampled municipalities are used in every test year and
every variant), 95% percentile interval of the difference against the reference. Differences of about one point
are only trusted if the interval excludes zero.

SHAP by family (last test year): LightGBM TreeSHAP (pred_contrib). In B the history is the offset, so its share is
the mean absolute deviation of the GLM linear predictor; the table also shows the shares WITHOUT the history.

Input : Data/Processed/panel_final_<panel>.csv (needs PESO_ESTUDIANTES and COD_MPIO_DANE)
Output: Data/Auditorio_Panel_Final/pre_modelo_experimento_offset_<panel>.csv and .md
        Data/Auditorio_Panel_Final/pre_modelo_experimento_offset_bootstrap_<panel>.csv
        Data/Auditorio_Panel_Final/pre_modelo_experimento_offset_shap_familias_<panel>.csv
=============================================================================
"""

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import PoissonRegressor

try:
    import lightgbm as lgb
except ImportError as err:  # corporate machines sometimes block pip: fail early with a clear message
    raise ImportError("lightgbm is required for this experiment (pip install lightgbm).") from err

# =============================================================================
# 0. PATHS
# =============================================================================
BASE_DIR = Path(
    r"C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data"
)
IN_DIR = BASE_DIR / "Processed"
AUDIT_DIR = BASE_DIR / "Auditorio_Panel_Final"

# =============================================================================
# 1. GENERAL VARIABLES
# =============================================================================
COL_SEDE, COL_ANIO, COL_MPIO = "SEDE_CODIGO", "ANIO_OBJETIVO", "COD_MPIO_DANE"
COL_RATE, COL_PESO, COL_SIZE = "TASA_DESERCION_SEDE", "PESO_ESTUDIANTES", "MATRICULA_TOTAL"
COL_HIST = "DESERCION_LAG3"
COL_HIST_ADJ = "DESERCION_LAG3_AJUSTADA"
COLS_HIST_COUNTS = ["ESTUDIANTES_LAG3", "DESERTORES_LAG3"]
COLS_HIST_ALL = [COL_HIST, COL_HIST_ADJ] + COLS_HIST_COUNTS
HIST_LAG = 3

PREDICTORES_COMUNES = [
    "DESERCION_LAG3", "MATRICULA_TOTAL", "MATRICULA_PCT_CAMBIO", "PROP_DESPLAZADOS",
    "PROP_LIMITACION", "PROP_ETNIA", "PROP_FLEXIBLE", "FLAG_PDET", "FLAG_ZOMAC",
]
PREDICTORES_ICFES = ["ICFES_PROM_PUNT_GLOBAL", "ICFES_PROM_INSE"]
PANELES = {
    "ofrecen11": PREDICTORES_COMUNES + PREDICTORES_ICFES,
    "Solo_Primaria": PREDICTORES_COMUNES,
}
FAMILIAS = {  # intervention-oriented families used for the SHAP shares
    "Historia propia": COLS_HIST_ALL,
    "Tamaño y dinámica de matrícula": ["MATRICULA_TOTAL", "MATRICULA_PCT_CAMBIO"],
    "Composición del alumnado": ["PROP_DESPLAZADOS", "PROP_LIMITACION", "PROP_ETNIA", "PROP_FLEXIBLE"],
    "Contexto territorial": ["FLAG_PDET", "FLAG_ZOMAC"],
    "ICFES": ["ICFES_PROM_PUNT_GLOBAL", "ICFES_PROM_INSE"],
}

TEST_YEARS = [2021, 2022, 2023]
ALPHAS = [0.0, 0.1, 0.2, 0.3, 1.0]   # Priority = rate x students ^ ALPHA
TOP_SHARE = 0.10
TREE_GRID = [150, 300]               # two capacities; the last one is the reference capacity
N_BOOT = 200                         # municipality bootstrap replicates (lower it for a quick run)
SHAP_MAX_ROWS = 20000
SEED = 42

HGB_PARAMS = dict(                   # identical to the previous experiment (reproduction check)
    learning_rate=0.05, max_iter=300, max_depth=4, min_samples_leaf=50,
    l2_regularization=1.0, random_state=42,
)
LGBM_PARAMS = dict(
    objective="poisson", learning_rate=0.05, num_leaves=15, max_depth=4, min_child_samples=50,
    reg_lambda=1.0, random_state=SEED, n_jobs=4, verbose=-1,
)
METRICS_BOOT = ["CAPTURA_DENTRO_ESTRATO_PROM", "CAPTURA_DENTRO_Q5", "DEVIANZA_POISSON"]
LOWER_IS_BETTER = {"DEVIANZA_POISSON"}

# =============================================================================
# 2. FUNCTION DEFINITIONS
# =============================================================================

def estimate_k_empirical_bayes(deserters, students) -> float:
    """Empirical-Bayes strength of the prior (K = alpha + beta of a Beta-Binomial), method of moments (Kleinman, 1973)."""
    d, n = np.asarray(deserters, float), np.asarray(students, float)
    ok = np.isfinite(d) & np.isfinite(n) & (n > 0)
    d, n = d[ok], n[ok]
    big_n = len(d)
    if big_n < 3 or n.sum() - big_n <= 0:
        return np.nan
    m = d.sum() / n.sum()
    if not 0 < m < 1:
        return np.nan
    s = np.sum(n * (d / n - m) ** 2)
    rho = float(np.clip((s / (m * (1 - m)) - (big_n - 1)) / (n.sum() - big_n), 1e-6, 0.99))
    return (1 - rho) / rho


def load_panel(nombre: str) -> pd.DataFrame:
    """Final panel plus the students and deserters of the same sede in t-3, joined by calendar year."""
    df = pd.read_csv(
        IN_DIR / f"panel_final_{nombre}.csv", dtype={COL_SEDE: str, COL_MPIO: str}, low_memory=False
    )
    for col in (COL_PESO, COL_MPIO):
        if col not in df.columns:
            raise KeyError(f"{col} is missing from panel_final_{nombre}.csv: rerun the master panel and Phase 2.")
    hist = df[[COL_SEDE, COL_ANIO, COL_RATE, COL_PESO]].copy()
    hist[COL_ANIO] = hist[COL_ANIO] + HIST_LAG
    hist = hist.rename(columns={COL_RATE: "_rate_lag3", COL_PESO: "ESTUDIANTES_LAG3"})
    df = df.merge(hist, on=[COL_SEDE, COL_ANIO], how="left", validate="many_to_one")
    df["DESERTORES_LAG3"] = np.round(df["_rate_lag3"] * df["ESTUDIANTES_LAG3"])  # NaN where there is no record

    # The rebuilt history must match the DESERCION_LAG3 column of the panel
    both = df["_rate_lag3"].notna() & df[COL_HIST].notna()
    if both.any() and not np.allclose(df.loc[both, "_rate_lag3"], df.loc[both, COL_HIST], atol=1e-9):
        raise ValueError("The rate rebuilt for t-3 does not match DESERCION_LAG3: check the panel.")
    return df


def population(df: pd.DataFrame) -> pd.DataFrame:
    """Eligible rows with a Sit_Acad record (students known); deserters rebuilt from rate x students."""
    pop = df[(df["ELEGIBLE_MODELO"] == 1) & df[COL_PESO].gt(0) & df[COL_RATE].notna()].copy()
    pop["_d"] = np.round(pop[COL_RATE] * pop[COL_PESO])
    return pop


def add_shrunk_history(train: pd.DataFrame, test: pd.DataFrame, k: float) -> tuple:
    """
    DESERCION_LAG3_AJUSTADA = (deserters_t-3 + K * mean) / (students_t-3 + K). K and the mean come from the TRAIN
    years only; a sede without counts in t-3 gets the prior mean. The value is always > 0, so its log is defined.
    """
    m_all = train["_d"].sum() / train[COL_PESO].sum()
    out = []
    for frame in (train, test):
        frame = frame.copy()
        known = frame["ESTUDIANTES_LAG3"].gt(0)
        shrunk = (frame["DESERTORES_LAG3"] + k * m_all) / (frame["ESTUDIANTES_LAG3"] + k)
        frame[COL_HIST_ADJ] = np.where(known, shrunk, m_all)
        out.append(frame)
    return out[0], out[1]


def build_variants(base_features: list) -> list:
    """Feature sets and settings of the LightGBM variants (A, B, B2, C) for each capacity."""
    non_hist = [c for c in base_features if c != COL_HIST]
    all_feats = base_features + [COL_HIST_ADJ] + COLS_HIST_COUNTS
    mono = [1 if c in (COL_HIST, COL_HIST_ADJ) else 0 for c in all_feats]
    specs = []
    for t in TREE_GRID:
        specs += [
            dict(name=f"A_features_t{t}", feats=all_feats, offset=False, mono=None, trees=t),
            dict(name=f"B_offset_sin_hist_t{t}", feats=non_hist, offset=True, mono=None, trees=t),
            dict(name=f"B2_offset_con_hist_t{t}", feats=all_feats, offset=True, mono=None, trees=t),
            dict(name=f"C_offset_con_hist_mono_t{t}", feats=all_feats, offset=True, mono=mono, trees=t),
        ]
    return specs


def fit_history_glm(train: pd.DataFrame) -> PoissonRegressor:
    """Poisson GLM, rate ~ exp(b0 + b1 * log(shrunk history)), students as weight (same as an offset on log students)."""
    glm = PoissonRegressor(alpha=0.0, max_iter=500)
    glm.fit(np.log(train[[COL_HIST_ADJ]].to_numpy()), train[COL_RATE].to_numpy(), sample_weight=train[COL_PESO].to_numpy())
    return glm


def glm_linear_predictor(glm: PoissonRegressor, frame: pd.DataFrame) -> np.ndarray:
    """Linear predictor (log scale) of the history GLM: this is the offset passed to the GBM."""
    return glm.intercept_ + glm.coef_[0] * np.log(frame[COL_HIST_ADJ].to_numpy())


def fit_predict_lgbm(train, test, spec, off_train, off_test) -> tuple:
    """Poisson LightGBM (rate target, students as weight); with offset the GLM predictor is init_score."""
    params = dict(LGBM_PARAMS, n_estimators=spec["trees"])
    if spec["mono"] is not None:
        params["monotone_constraints"] = spec["mono"]
    mdl = lgb.LGBMRegressor(**params)
    fit_kwargs = dict(sample_weight=train[COL_PESO].to_numpy())
    if spec["offset"]:
        fit_kwargs["init_score"] = off_train
    mdl.fit(train[spec["feats"]], train[COL_RATE].to_numpy(), **fit_kwargs)
    raw = mdl.predict(test[spec["feats"]], raw_score=True)  # log scale, WITHOUT the offset
    if spec["offset"]:
        raw = raw + off_test
    return mdl, np.exp(raw)


def poisson_deviance(d: np.ndarray, mu: np.ndarray) -> float:
    """Mean Poisson deviance of the expected number of deserters (lower is better)."""
    mu = np.clip(mu, 1e-9, None)
    term = np.where(d > 0, d * np.log(np.where(d > 0, d, 1.0) / mu), 0.0)
    return float(np.mean(2 * (term - (d - mu))))


def capture_by_stratum(d, rate_hat, quintil, jitter, share=TOP_SHARE) -> list:
    """Inside each size quintile, % of the stratum's deserters in the top `share` of its sedes by predicted rate (random = 10%)."""
    caps = []
    for qn in range(5):
        idx = np.flatnonzero(quintil == qn)
        if idx.size == 0 or d[idx].sum() == 0:
            caps.append(np.nan)
            continue
        k = int(np.ceil(share * idx.size))
        top = idx[np.argsort(-(rate_hat[idx] + jitter[idx]))[:k]]
        caps.append(100 * d[top].sum() / d[idx].sum())
    return caps


def evaluate(test, rate_hat, label, anio, panel, quintil, n_pred, jitter) -> list:
    """Capture of deserters, fairness to small sedes, discrimination net of size, calibration and dispersion."""
    d = test["_d"].to_numpy()
    mu = rate_hat * test[COL_PESO].to_numpy()
    intra = capture_by_stratum(d, rate_hat, quintil, jitter)
    top_mu = mu >= np.quantile(mu, 0.90)
    pearson = float(np.mean((d - mu) ** 2 / np.clip(mu, 1e-2, None)))  # approximate: mu floored at 0.01
    filas = []
    for alpha in ALPHAS:
        score = rate_hat * n_pred ** alpha + jitter
        top = np.zeros(len(d), bool)
        top[np.argsort(-score)[: int(np.ceil(TOP_SHARE * len(d)))]] = True
        filas.append({
            "PANEL": panel, "ANIO_TEST": anio, "CONJUNTO": label, "ALPHA": alpha,
            "CAPTURA_DESERTORES": 100 * d[top].sum() / d.sum() if d.sum() else np.nan,
            "CAPTURA_Q1_PEQUENAS": 100 * float(d[top & (quintil == 0)].sum() / max(d[quintil == 0].sum(), 1)),
            "SPEARMAN": spearmanr(score, d)[0],
            "CALIBRACION": float(mu.sum() / d.sum()) if d.sum() else np.nan,
            "CALIB_TOP10_MU": float(d[top_mu].sum() / max(mu[top_mu].sum(), 1e-9)),
            "DEVIANZA_POISSON": poisson_deviance(d, mu),
            "PEARSON_APROX": pearson,
            "CAPTURA_DENTRO_ESTRATO_PROM": float(np.nanmean(intra)),
            "CAPTURA_DENTRO_Q1": intra[0], "CAPTURA_DENTRO_Q5": intra[4],
        })
    return filas


def core_metrics(d, rate_hat, peso, quintil, jitter) -> np.ndarray:
    """The three metrics used in the bootstrap: mean capture net of size, capture in Q5, Poisson deviance."""
    intra = capture_by_stratum(d, rate_hat, quintil, jitter)
    return np.array([np.nanmean(intra), intra[4], poisson_deviance(d, rate_hat * peso)])


def mean_core_metrics(store: dict, variants: list, counts=None) -> np.ndarray:
    """Core metrics averaged over the test years; `counts` = bootstrap multiplicity of each municipality (None = original data)."""
    out = np.zeros((len(variants), len(METRICS_BOOT)))
    for s in store.values():
        idx = np.arange(len(s["d"])) if counts is None else np.repeat(np.arange(len(s["d"])), counts[s["mpio_code"]])
        d, peso, quintil, jitter = s["d"][idx], s["peso"][idx], s["quintil"][idx], s["jitter"][idx]
        for j, v in enumerate(variants):
            out[j] += core_metrics(d, s["preds"][v][idx], peso, quintil, jitter)
    return out / len(store)


def bootstrap_pairs(store: dict, pairs: list, n_boot: int, panel: str) -> pd.DataFrame:
    """Paired bootstrap by municipality: difference (first - second) of the core metrics with a 95% percentile interval."""
    rng = np.random.default_rng(SEED)
    mpios = pd.unique(np.concatenate([s["mpio"] for s in store.values()]))
    code = pd.Series(np.arange(len(mpios)), index=mpios)
    for s in store.values():
        s["mpio_code"] = pd.Series(s["mpio"]).map(code).to_numpy()
    variants = sorted({v for pair in pairs for v in pair})
    pos = {v: i for i, v in enumerate(variants)}
    point = mean_core_metrics(store, variants)
    reps = np.zeros((n_boot, len(variants), len(METRICS_BOOT)))
    for b in range(n_boot):
        counts = np.bincount(rng.integers(0, len(mpios), len(mpios)), minlength=len(mpios))
        reps[b] = mean_core_metrics(store, variants, counts)
        if (b + 1) % 50 == 0:
            print(f"  bootstrap {panel}: {b + 1}/{n_boot}")
    filas = []
    for a, ref in pairs:
        for m, metric in enumerate(METRICS_BOOT):
            diff = reps[:, pos[a], m] - reps[:, pos[ref], m]
            better = (diff < 0) if metric in LOWER_IS_BETTER else (diff > 0)
            filas.append({
                "PANEL": panel, "VARIANTE": a, "REFERENCIA": ref, "METRICA": metric,
                "DIF_PUNTUAL": point[pos[a], m] - point[pos[ref], m],
                "IC95_BAJO": np.nanpercentile(diff, 2.5), "IC95_ALTO": np.nanpercentile(diff, 97.5),
                "P_MEJOR": float(np.nanmean(better)),
            })
    return pd.DataFrame(filas)


def family_shares(mdl, test: pd.DataFrame, spec: dict, glm) -> dict:
    """Mean |contribution| by family (TreeSHAP of the GBM; the offset history is added as the |deviation| of the GLM predictor)."""
    sample = test.sample(min(SHAP_MAX_ROWS, len(test)), random_state=SEED)
    contrib = mdl.predict(sample[spec["feats"]], pred_contrib=True)[:, :-1]
    mean_abs = dict(zip(spec["feats"], np.abs(contrib).mean(axis=0)))
    shares = {fam: float(sum(v for c, v in mean_abs.items() if c in cols)) for fam, cols in FAMILIAS.items()}
    if spec["offset"]:
        lin = glm_linear_predictor(glm, sample)
        shares["Historia propia"] += float(np.abs(lin - lin.mean()).mean())
    total = sum(shares.values())
    no_hist = total - shares["Historia propia"]
    out = {}
    for fam, v in shares.items():
        out[fam] = {"PCT_TOTAL": 100 * v / total, "PCT_SIN_HISTORIA": np.nan if fam == "Historia propia" else 100 * v / no_hist}
    return out


def to_markdown(df: pd.DataFrame, nd: int = 2) -> str:
    """Minimal markdown table (no extra dependency)."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for r in df.itertuples(index=False):
        lines.append("| " + " | ".join(f"{v:.{nd}f}" if isinstance(v, float) else str(v) for v in r) + " |")
    return "\n".join(lines)


def save_report(tabla, boot, shap_tab, nombre) -> None:
    """Write csv and md; a locked OneDrive file must not abort the experiment."""
    try:
        AUDIT_DIR.mkdir(parents=True, exist_ok=True)
        stem = f"pre_modelo_experimento_offset_{nombre}"
        tabla.to_csv(AUDIT_DIR / f"{stem}.csv", index=False)
        boot.to_csv(AUDIT_DIR / f"pre_modelo_experimento_offset_bootstrap_{nombre}.csv", index=False)
        shap_tab.to_csv(AUDIT_DIR / f"pre_modelo_experimento_offset_shap_familias_{nombre}.csv", index=False)
        base = tabla[tabla["ALPHA"] == ALPHAS[0]]
        intra = base.groupby("CONJUNTO")[
            ["CAPTURA_DENTRO_ESTRATO_PROM", "CAPTURA_DENTRO_Q1", "CAPTURA_DENTRO_Q5", "DEVIANZA_POISSON",
             "CALIBRACION", "CALIB_TOP10_MU", "PEARSON_APROX"]
        ].mean().reset_index().sort_values("CAPTURA_DENTRO_ESTRATO_PROM", ascending=False)
        por_anio = base.pivot_table(index="CONJUNTO", columns="ANIO_TEST", values="CAPTURA_DENTRO_ESTRATO_PROM").reset_index()
        por_anio.columns = [str(c) for c in por_anio.columns]
        listas = tabla.groupby(["CONJUNTO", "ALPHA"])[["CAPTURA_DESERTORES", "CAPTURA_Q1_PEQUENAS", "SPEARMAN"]].mean().reset_index()
        md = [
            f"# Experimento de offset - panel {nombre}",
            "\n## Discriminación dentro de cada estrato de tamaño, devianza y calibración (promedio de los años; azar = 10%)\n",
            "CALIBRACION ideal = 1; CALIB_TOP10_MU = observados / esperados en el 10% de mayor mu; PEARSON_APROX cercano a 1 = sin sobredispersión (mu con piso de 0,01).\n",
            to_markdown(intra),
            "\n## Captura dentro del estrato por año de prueba\n",
            to_markdown(por_anio),
            f"\n## Diferencias pareadas con IC 95% (bootstrap por municipio, {N_BOOT} réplicas; positivo = mejor salvo DEVIANZA_POISSON, donde negativo = mejor)\n",
            "P_MEJOR = proporción de réplicas en que la variante supera a la referencia.\n",
            to_markdown(boot.drop(columns=["PANEL"]), nd=3),
            "\n## Participación por familia (último año de prueba; % de la contribución absoluta media)\n",
            to_markdown(shap_tab.drop(columns=["PANEL"])),
            "\n## Lista del 10% de sedes por variante y alpha (promedio de los años)\n",
            to_markdown(listas),
        ]
        (AUDIT_DIR / f"{stem}.md").write_text("\n".join(md), encoding="utf-8")
    except OSError as err:
        print(f"[WARN] Could not write the report of {nombre}: {err}")


def run_panel(nombre: str, base_features: list) -> tuple:
    """Rolling-origin backtests for one panel: baselines + LightGBM variants A, B, B2, C (two capacities each)."""
    pop = population(load_panel(nombre))
    specs = build_variants(base_features)
    rng = np.random.default_rng(SEED)
    filas, store, shap_rows = [], {}, []
    for anio in TEST_YEARS:
        train, test = pop[pop[COL_ANIO] < anio], pop[pop[COL_ANIO] == anio]
        if train.empty or test.empty:
            print(f"[WARN] {nombre} {anio}: no train or test rows, skipped.")
            continue
        k_eb = estimate_k_empirical_bayes(train["_d"], train[COL_PESO])
        print(f"{nombre} | test {anio} | train rows {len(train):,} | test rows {len(test):,} | K empirical Bayes = {k_eb:.1f}")
        train, test = add_shrunk_history(train, test, k_eb)
        glm = fit_history_glm(train)
        off_train, off_test = glm_linear_predictor(glm, train), glm_linear_predictor(glm, test)
        print(f"  history GLM: intercept = {glm.intercept_:.3f}, slope on log(shrunk history) = {glm.coef_[0]:.3f}")

        n_pred = test[COL_SIZE].fillna(test[COL_SIZE].median()).clip(lower=1).to_numpy()
        quintil = pd.qcut(pd.Series(n_pred).rank(method="first"), 5, labels=False).to_numpy()
        jitter = rng.random(len(test)) * 1e-12
        store[anio] = dict(
            d=test["_d"].to_numpy(), peso=test[COL_PESO].to_numpy(), quintil=quintil, jitter=jitter,
            mpio=test[COL_MPIO].astype(str).to_numpy(), preds={},
        )

        def register(label: str, rate_hat: np.ndarray) -> None:
            filas.extend(evaluate(test, rate_hat, label, anio, nombre, quintil, n_pred, jitter))
            store[anio]["preds"][label] = rate_hat

        # Baselines
        register("B0b_historia_ajustada", test[COL_HIST_ADJ].to_numpy())
        register("GLM_solo_historia", np.exp(off_test))
        hgb = HistGradientBoostingRegressor(loss="poisson", **HGB_PARAMS)  # reproduction of the previous M4_BASE
        hgb.fit(train[base_features], train[COL_RATE], sample_weight=train[COL_PESO])
        register("M4_BASE_HGB", hgb.predict(test[base_features]))

        # LightGBM variants
        for spec in specs:
            mdl, rate_hat = fit_predict_lgbm(train, test, spec, off_train, off_test)
            register(spec["name"], rate_hat)
            if anio == TEST_YEARS[-1] and spec["trees"] == TREE_GRID[-1] and not spec["name"].startswith("C_"):
                for fam, vals in family_shares(mdl, test, spec, glm).items():
                    shap_rows.append({"PANEL": nombre, "VARIANTE": spec["name"], "FAMILIA": fam, **vals})
    tabla = pd.DataFrame(filas)
    ref = f"A_features_t{TREE_GRID[-1]}"
    names = [s["name"] for s in specs] + ["B0b_historia_ajustada", "GLM_solo_historia", "M4_BASE_HGB"]
    pairs = [(v, ref) for v in names if v != ref]
    pairs += [(f"B_offset_sin_hist_t{t}", f"B2_offset_con_hist_t{t}") for t in TREE_GRID]   # cost of leaving the lag out
    pairs += [(f"{p}_t{TREE_GRID[-1]}", "GLM_solo_historia") for p in ("B_offset_sin_hist", "B2_offset_con_hist")]  # added value of the GBM
    return tabla, store, pairs, pd.DataFrame(shap_rows)


# =============================================================================
# 3. EXECUTION
# =============================================================================

def main() -> None:
    for nombre, features in PANELES.items():
        tabla, store, pairs, shap_tab = run_panel(nombre, features)
        boot = bootstrap_pairs(store, pairs, N_BOOT, nombre)
        save_report(tabla, boot, shap_tab, nombre)
        print(f"\n=== {nombre}: discrimination net of size (mean over test years; random = 10%) ===")
        print(
            tabla[tabla["ALPHA"] == ALPHAS[0]].groupby("CONJUNTO")[
                ["CAPTURA_DENTRO_ESTRATO_PROM", "CAPTURA_DENTRO_Q1", "CAPTURA_DENTRO_Q5", "DEVIANZA_POISSON", "CALIBRACION"]
            ].mean().round(2).sort_values("CAPTURA_DENTRO_ESTRATO_PROM", ascending=False).to_string()
        )
        print(f"\n=== {nombre}: paired differences with 95% bootstrap interval (by municipality) ===")
        print(boot.drop(columns=["PANEL"]).round(3).to_string(index=False))
        print(f"\n=== {nombre}: share by family, last test year ===")
        print(shap_tab.pivot_table(index="FAMILIA", columns="VARIANTE", values="PCT_TOTAL").round(1).to_string())


if __name__ == "__main__":
    main()


In [ ]:
"""
=============================================================================
PRE-MODELING 5: frontier, capacity, saturated weights and calibration by year
=============================================================================
Follow-up of the offset experiment. Four open points, one block each:

1. FRONTIER. Comparing models at a fixed alpha is unfair: a model with flatter predicted rates (the history-only GLM)
   is dominated by n^alpha much sooner and stops listing small sedes. Here every model is swept over a fine grid of
   alpha and compared at the SAME coverage of the smallest quintile (Q1): "how much of the total do you capture
   when the list still catches 10% / 20% of the Q1 deserters?".
2. UNCERTAINTY. Paired bootstrap by municipality (same draw for every year and every variant) with capture in each
   size quintile (Q1..Q5), and a sensitivity that keeps only the test years 2022-2023 (the 2021 fold trains on a
   single year and is the weakest for every GBM).
3. CAPACITY and WEIGHTS. Three capacities (c1 < c2 < c3, c3 = the capacity of experiment 4) and two weightings:
   wn  = students (as before) and wsat = n*K/(n+K), a saturated weight. With K ~ 10 (empirical Bayes, train years only),
   a big sede carries much less information than its size says (overdispersion), so it should not dominate the fit.
4. CALIBRATION by year, plus a "recalibration with the previous test year" check (observed/expected of the previous
   year used as a scale factor), to see whether the over-prediction is a time drift.

Model: the design chosen in experiment 4 (B2): GLM on log(shrunk history) as offset + LightGBM Poisson that also sees
the history features (explain it later WITHOUT the history family). Sanity check: B2_wn_c3 must reproduce
B2_offset_con_hist_t150 of experiment 4 (about 24.0 in Solo_Primaria and 23.1 in ofrecen11, capture net of size).

Caveat: the grid is evaluated on the same backtest folds, so it points the direction; the final hyperparameters must
be tuned with nested validation by year.

Input : Data/Processed/panel_final_<panel>.csv
Output: Data/Auditorio_Panel_Final/pre_modelo_experimento_frontera_capacidad_<panel>.md  (main report)
        ..._bootstrap_<panel>.csv, ..._frontera_<panel>.csv, ..._calibracion_<panel>.csv, ..._frontera_<panel>.png
=============================================================================
"""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import PoissonRegressor

try:
    import lightgbm as lgb
except ImportError as err:  # corporate machines sometimes block pip: fail early with a clear message
    raise ImportError("lightgbm is required for this experiment (pip install lightgbm).") from err

# =============================================================================
# 0. PATHS
# =============================================================================
BASE_DIR = Path(
    r"C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data"
)
IN_DIR = BASE_DIR / "Processed"
AUDIT_DIR = BASE_DIR / "Auditorio_Panel_Final"

# =============================================================================
# 1. GENERAL VARIABLES
# =============================================================================
COL_SEDE, COL_ANIO, COL_MPIO = "SEDE_CODIGO", "ANIO_OBJETIVO", "COD_MPIO_DANE"
COL_RATE, COL_PESO, COL_SIZE = "TASA_DESERCION_SEDE", "PESO_ESTUDIANTES", "MATRICULA_TOTAL"
COL_HIST = "DESERCION_LAG3"
COL_HIST_ADJ = "DESERCION_LAG3_AJUSTADA"
COLS_HIST_COUNTS = ["ESTUDIANTES_LAG3", "DESERTORES_LAG3"]
HIST_LAG = 3

PREDICTORES_COMUNES = [
    "DESERCION_LAG3", "MATRICULA_TOTAL", "MATRICULA_PCT_CAMBIO", "PROP_DESPLAZADOS",
    "PROP_LIMITACION", "PROP_ETNIA", "PROP_FLEXIBLE", "FLAG_PDET", "FLAG_ZOMAC",
]
PREDICTORES_ICFES = ["ICFES_PROM_PUNT_GLOBAL", "ICFES_PROM_INSE"]
PANELES = {
    "ofrecen11": PREDICTORES_COMUNES + PREDICTORES_ICFES,
    "Solo_Primaria": PREDICTORES_COMUNES,
}

TEST_YEARS = [2021, 2022, 2023]
SENSITIVITY_YEARS = [2022, 2023]
TOP_SHARE = 0.10
FRONT_ALPHAS = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.6, 1.0]
Q1_TARGETS = [10, 20]                      # % of the Q1 deserters that the list must still catch
WEIGHT_SCHEMES = ["n", "sat"]              # n = students; sat = n*K/(n+K)
CAPACITIES = {                             # c3 = capacity of experiment 4 (150 trees, min_child_samples 50)
    "c1": dict(trees=50, mcs=300),
    "c2": dict(trees=100, mcs=150),
    "c3": dict(trees=150, mcs=50),
}
N_BOOT = 200                               # municipality bootstrap replicates (lower it for a quick run)
SEED = 42

LGBM_PARAMS = dict(
    objective="poisson", learning_rate=0.05, num_leaves=15, max_depth=4, reg_lambda=1.0,
    random_state=SEED, n_jobs=4, verbose=-1,
)
METRIC_NAMES = (
    ["CAPT_Q1", "CAPT_Q2", "CAPT_Q3", "CAPT_Q4", "CAPT_Q5", "CAPT_PROM", "DEVIANZA"]
    + [f"TOTAL_CON_Q1_{t}" for t in Q1_TARGETS]
)
METRICS_IN_MD = ["CAPT_Q1", "CAPT_Q5", "CAPT_PROM", "DEVIANZA"] + [f"TOTAL_CON_Q1_{t}" for t in Q1_TARGETS]
LOWER_IS_BETTER = {"DEVIANZA"}

# =============================================================================
# 2. FUNCTION DEFINITIONS
# =============================================================================

def estimate_k_empirical_bayes(deserters, students) -> float:
    """Empirical-Bayes strength of the prior (K = alpha + beta of a Beta-Binomial), method of moments (Kleinman, 1973)."""
    d, n = np.asarray(deserters, float), np.asarray(students, float)
    ok = np.isfinite(d) & np.isfinite(n) & (n > 0)
    d, n = d[ok], n[ok]
    big_n = len(d)
    if big_n < 3 or n.sum() - big_n <= 0:
        return np.nan
    m = d.sum() / n.sum()
    if not 0 < m < 1:
        return np.nan
    s = np.sum(n * (d / n - m) ** 2)
    rho = float(np.clip((s / (m * (1 - m)) - (big_n - 1)) / (n.sum() - big_n), 1e-6, 0.99))
    return (1 - rho) / rho


def load_panel(nombre: str) -> pd.DataFrame:
    """Final panel plus the students and deserters of the same sede in t-3, joined by calendar year."""
    df = pd.read_csv(
        IN_DIR / f"panel_final_{nombre}.csv", dtype={COL_SEDE: str, COL_MPIO: str}, low_memory=False
    )
    for col in (COL_PESO, COL_MPIO):
        if col not in df.columns:
            raise KeyError(f"{col} is missing from panel_final_{nombre}.csv: rerun the master panel and Phase 2.")
    hist = df[[COL_SEDE, COL_ANIO, COL_RATE, COL_PESO]].copy()
    hist[COL_ANIO] = hist[COL_ANIO] + HIST_LAG
    hist = hist.rename(columns={COL_RATE: "_rate_lag3", COL_PESO: "ESTUDIANTES_LAG3"})
    df = df.merge(hist, on=[COL_SEDE, COL_ANIO], how="left", validate="many_to_one")
    df["DESERTORES_LAG3"] = np.round(df["_rate_lag3"] * df["ESTUDIANTES_LAG3"])  # NaN where there is no record
    both = df["_rate_lag3"].notna() & df[COL_HIST].notna()
    if both.any() and not np.allclose(df.loc[both, "_rate_lag3"], df.loc[both, COL_HIST], atol=1e-9):
        raise ValueError("The rate rebuilt for t-3 does not match DESERCION_LAG3: check the panel.")
    return df


def population(df: pd.DataFrame) -> pd.DataFrame:
    """Eligible rows with a Sit_Acad record (students known); deserters rebuilt from rate x students."""
    pop = df[(df["ELEGIBLE_MODELO"] == 1) & df[COL_PESO].gt(0) & df[COL_RATE].notna()].copy()
    pop["_d"] = np.round(pop[COL_RATE] * pop[COL_PESO])
    return pop


def add_shrunk_history(train: pd.DataFrame, test: pd.DataFrame, k: float) -> tuple:
    """DESERCION_LAG3_AJUSTADA = (deserters_t-3 + K*mean) / (students_t-3 + K); K and mean from TRAIN only, always > 0."""
    m_all = train["_d"].sum() / train[COL_PESO].sum()
    out = []
    for frame in (train, test):
        frame = frame.copy()
        known = frame["ESTUDIANTES_LAG3"].gt(0)
        shrunk = (frame["DESERTORES_LAG3"] + k * m_all) / (frame["ESTUDIANTES_LAG3"] + k)
        frame[COL_HIST_ADJ] = np.where(known, shrunk, m_all)
        out.append(frame)
    return out[0], out[1]


def make_weights(frame: pd.DataFrame, scheme: str, k: float) -> np.ndarray:
    """Fit weights: students ('n') or saturated students n*K/(n+K) ('sat'), which caps the influence of very large sedes."""
    n = frame[COL_PESO].to_numpy(float)
    return n if scheme == "n" else n * k / (n + k)


def fit_history_glm(train: pd.DataFrame, weights: np.ndarray) -> PoissonRegressor:
    """Poisson GLM, rate ~ exp(b0 + b1 * log(shrunk history)); the weights play the role of the exposure."""
    glm = PoissonRegressor(alpha=0.0, max_iter=500)
    glm.fit(np.log(train[[COL_HIST_ADJ]].to_numpy()), train[COL_RATE].to_numpy(), sample_weight=weights)
    return glm


def glm_linear_predictor(glm: PoissonRegressor, frame: pd.DataFrame) -> np.ndarray:
    """Linear predictor (log scale) of the history GLM: this is the offset passed to the GBM."""
    return glm.intercept_ + glm.coef_[0] * np.log(frame[COL_HIST_ADJ].to_numpy())


def fit_predict_lgbm(train, test, feats, cap, weights, off_train, off_test) -> np.ndarray:
    """Poisson LightGBM with the GLM predictor as init_score; returns the predicted rate (offset added back by hand)."""
    mdl = lgb.LGBMRegressor(**LGBM_PARAMS, n_estimators=cap["trees"], min_child_samples=cap["mcs"])
    mdl.fit(train[feats], train[COL_RATE].to_numpy(), sample_weight=weights, init_score=off_train)
    return np.exp(mdl.predict(test[feats], raw_score=True) + off_test)


def poisson_deviance(d: np.ndarray, mu: np.ndarray) -> float:
    """Mean Poisson deviance of the expected number of deserters (lower is better)."""
    mu = np.clip(mu, 1e-9, None)
    term = np.where(d > 0, d * np.log(np.where(d > 0, d, 1.0) / mu), 0.0)
    return float(np.mean(2 * (term - (d - mu))))


def capture_by_stratum(d, rate_hat, quintil, jitter, share=TOP_SHARE) -> list:
    """Inside each size quintile, % of the stratum's deserters in the top `share` of its sedes by predicted rate (random = 10%)."""
    caps = []
    for qn in range(5):
        idx = np.flatnonzero(quintil == qn)
        if idx.size == 0 or d[idx].sum() == 0:
            caps.append(np.nan)
            continue
        k = int(np.ceil(share * idx.size))
        top = idx[np.argsort(-(rate_hat[idx] + jitter[idx]))[:k]]
        caps.append(100 * d[top].sum() / d[idx].sum())
    return caps


def frontier_points(d, rate_hat, n_pows, quintil, jitter) -> tuple:
    """For each alpha: (% of the Q1 deserters caught, % of ALL deserters caught) by the top-10% list ranked by rate x n^alpha."""
    k = int(np.ceil(TOP_SHARE * len(d)))
    q1 = quintil == 0
    d_tot, d_q1 = d.sum(), d[q1].sum()
    xs, ys = [], []
    for n_pow in n_pows:
        top = np.argpartition(-(rate_hat * n_pow + jitter), k - 1)[:k]
        ys.append(100 * d[top].sum() / d_tot)
        xs.append(100 * d[top][quintil[top] == 0].sum() / d_q1 if d_q1 > 0 else np.nan)
    return np.array(xs), np.array(ys)


def year_metrics(d, rate_hat, peso, quintil, n_pows, jitter) -> np.ndarray:
    """Capture in each size quintile, their mean, Poisson deviance and total capture at fixed Q1 coverage (interpolated frontier)."""
    caps = capture_by_stratum(d, rate_hat, quintil, jitter)
    xs, ys = frontier_points(d, rate_hat, n_pows, quintil, jitter)
    order = np.argsort(xs)
    totals = [float(np.interp(t, xs[order], ys[order])) for t in Q1_TARGETS]
    return np.array(caps + [np.nanmean(caps), poisson_deviance(d, rate_hat * peso)] + totals)


def all_year_metrics(store: dict, variants: list, counts=None) -> dict:
    """Core metrics per test year; `counts` = bootstrap multiplicity of each municipality (None = original data)."""
    out = {}
    for anio, s in store.items():
        idx = np.arange(len(s["d"])) if counts is None else np.repeat(np.arange(len(s["d"])), counts[s["mpio_code"]])
        d, peso, quintil, jitter = s["d"][idx], s["peso"][idx], s["quintil"][idx], s["jitter"][idx]
        n_pows = [s["n_pred"][idx] ** a for a in FRONT_ALPHAS]  # shared by all variants
        out[anio] = np.vstack([year_metrics(d, s["preds"][v][idx], peso, quintil, n_pows, jitter) for v in variants])
    return out


def scope_mean(year_vals: dict, years: list) -> np.ndarray:
    """Average of the per-year metric matrices over the chosen years."""
    return np.mean([year_vals[y] for y in years], axis=0)


def bootstrap_pairs(store: dict, pairs: list, n_boot: int, panel: str) -> tuple:
    """Paired bootstrap by municipality; returns (long table of differences, point-estimate table per variant and scope)."""
    rng = np.random.default_rng(SEED)
    mpios = pd.unique(np.concatenate([s["mpio"] for s in store.values()]))
    code = pd.Series(np.arange(len(mpios)), index=mpios)
    for s in store.values():
        s["mpio_code"] = pd.Series(s["mpio"]).map(code).to_numpy()
    variants = sorted({v for pair in pairs for v in pair})
    pos = {v: i for i, v in enumerate(variants)}
    scopes = {"TODOS": list(store), "2022-2023": [y for y in SENSITIVITY_YEARS if y in store]}
    scopes = {k: v for k, v in scopes.items() if v}
    point = {sc: scope_mean(all_year_metrics(store, variants), yrs) for sc, yrs in scopes.items()}
    reps = {sc: np.zeros((n_boot, len(variants), len(METRIC_NAMES))) for sc in scopes}
    for b in range(n_boot):
        counts = np.bincount(rng.integers(0, len(mpios), len(mpios)), minlength=len(mpios))
        year_vals = all_year_metrics(store, variants, counts)
        for sc, yrs in scopes.items():
            reps[sc][b] = scope_mean(year_vals, yrs)
        if (b + 1) % 50 == 0:
            print(f"  bootstrap {panel}: {b + 1}/{n_boot}")
    filas = []
    for sc in scopes:
        for a, ref in pairs:
            for m, metric in enumerate(METRIC_NAMES):
                diff = reps[sc][:, pos[a], m] - reps[sc][:, pos[ref], m]
                better = (diff < 0) if metric in LOWER_IS_BETTER else (diff > 0)
                filas.append({
                    "PANEL": panel, "ALCANCE": sc, "VARIANTE": a, "REFERENCIA": ref, "METRICA": metric,
                    "DIF_PUNTUAL": point[sc][pos[a], m] - point[sc][pos[ref], m],
                    "IC95_BAJO": np.nanpercentile(diff, 2.5), "IC95_ALTO": np.nanpercentile(diff, 97.5),
                    "P_MEJOR": float(np.nanmean(better)),
                })
    resumen = pd.concat(
        [pd.DataFrame(point[sc], index=pd.Index(variants, name="CONJUNTO"), columns=METRIC_NAMES).assign(ALCANCE=sc).reset_index()
         for sc in scopes]
    )
    return pd.DataFrame(filas), resumen


def frontier_table(store: dict, variants: list, panel: str) -> pd.DataFrame:
    """Frontier points (Q1 coverage vs total capture) per variant and alpha, averaged over the test years."""
    filas = []
    for anio, s in store.items():
        n_pows = [s["n_pred"] ** a for a in FRONT_ALPHAS]
        for v in variants:
            xs, ys = frontier_points(s["d"], s["preds"][v], n_pows, s["quintil"], s["jitter"])
            filas += [
                {"PANEL": panel, "ANIO_TEST": anio, "CONJUNTO": v, "ALPHA": a, "CAPTURA_Q1": x, "CAPTURA_TOTAL": y}
                for a, x, y in zip(FRONT_ALPHAS, xs, ys)
            ]
    return pd.DataFrame(filas).groupby(["PANEL", "CONJUNTO", "ALPHA"], as_index=False)[["CAPTURA_Q1", "CAPTURA_TOTAL"]].mean()


def calibration_table(store: dict, variants: list, panel: str) -> pd.DataFrame:
    """Expected/observed deserters per year, and the same after rescaling with the previous test year's observed/expected."""
    filas, prev_calib = [], {}
    for anio in sorted(store):
        s = store[anio]
        for v in variants:
            mu = s["preds"][v] * s["peso"]
            calib = float(mu.sum() / s["d"].sum())
            factor = 1.0 / prev_calib[v] if v in prev_calib else np.nan   # scale learned in the previous test year
            filas.append({
                "PANEL": panel, "ANIO_TEST": anio, "CONJUNTO": v, "CALIBRACION": calib,
                "CALIBRACION_RECAL": calib * factor, "DEVIANZA": poisson_deviance(s["d"], mu),
                "DEVIANZA_RECAL": poisson_deviance(s["d"], mu * factor) if np.isfinite(factor) else np.nan,
            })
            prev_calib[v] = calib
    return pd.DataFrame(filas)


def to_markdown(df: pd.DataFrame, nd: int = 2) -> str:
    """Minimal markdown table (no extra dependency)."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for r in df.itertuples(index=False):
        lines.append("| " + " | ".join(f"{v:.{nd}f}" if isinstance(v, float) else str(v) for v in r) + " |")
    return "\n".join(lines)


def diff_pivot(boot: pd.DataFrame, scope: str) -> pd.DataFrame:
    """One row per comparison: 'difference [95% interval]' per metric; * = interval excludes zero."""
    filas = []
    for (a, ref), g in boot[boot["ALCANCE"] == scope].groupby(["VARIANTE", "REFERENCIA"], sort=False):
        row = {"COMPARACION": f"{a} vs {ref}"}
        for r in g.itertuples():
            if r.METRICA not in METRICS_IN_MD:
                continue
            nd = 3 if r.METRICA == "DEVIANZA" else 2
            sig = "*" if (r.IC95_BAJO > 0 or r.IC95_ALTO < 0) else ""
            row[r.METRICA] = f"{r.DIF_PUNTUAL:+.{nd}f} [{r.IC95_BAJO:+.{nd}f}, {r.IC95_ALTO:+.{nd}f}]{sig}"
        filas.append(row)
    return pd.DataFrame(filas)


def plot_frontier(front: pd.DataFrame, nombre: str) -> None:
    """Frontier plot (x = % of Q1 deserters caught, y = % of all deserters caught); optional, skipped if matplotlib is missing."""
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    fig, ax = plt.subplots(figsize=(8, 6))
    for v, g in front.groupby("CONJUNTO"):
        style = "--" if v.startswith("GLM") else "-"
        ax.plot(g["CAPTURA_Q1"], g["CAPTURA_TOTAL"], style, marker="o", ms=3, label=v)
    ax.set_xlabel("% of the Q1 (smallest sedes) deserters caught by the top-10% list")
    ax.set_ylabel("% of ALL deserters caught by the top-10% list")
    ax.set_title(f"Frontier - {nombre} (mean of the test years; up and to the right is better)")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
    fig.savefig(AUDIT_DIR / f"pre_modelo_experimento_frontera_capacidad_frontera_{nombre}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)


def save_report(nombre, boot, resumen, front, calib, params, observed) -> None:
    """Write the md report and the csv files; a locked OneDrive file must not abort the experiment."""
    try:
        AUDIT_DIR.mkdir(parents=True, exist_ok=True)
        stem = f"pre_modelo_experimento_frontera_capacidad_{nombre}"
        boot.to_csv(AUDIT_DIR / f"{stem}_bootstrap.csv", index=False)
        front.to_csv(AUDIT_DIR / f"{stem}_frontera.csv", index=False)
        calib.to_csv(AUDIT_DIR / f"{stem}_calibracion.csv", index=False)
        cal_piv = calib.pivot_table(index="CONJUNTO", columns="ANIO_TEST", values="CALIBRACION").reset_index()
        rec_piv = calib.pivot_table(index="CONJUNTO", columns="ANIO_TEST", values="CALIBRACION_RECAL").reset_index()
        dev_piv = calib.pivot_table(index="CONJUNTO", columns="ANIO_TEST", values="DEVIANZA").reset_index()
        dev_rec = calib.pivot_table(index="CONJUNTO", columns="ANIO_TEST", values="DEVIANZA_RECAL").reset_index()
        for piv in (cal_piv, rec_piv, dev_piv, dev_rec):
            piv.columns = [str(c) for c in piv.columns]
        md = [
            f"# Experimento 5 (frontera, capacidad, pesos, calibración) - panel {nombre}",
            "\n## Parámetros por pliegue (K empírico Bayes y GLM de historia)\n",
            to_markdown(params, nd=3),
            "\n## Tasa observada por año (población elegible con registro)\n",
            to_markdown(observed, nd=4),
        ]
        for sc in resumen["ALCANCE"].unique():
            cols = ["CONJUNTO"] + METRIC_NAMES
            md += [f"\n## Resumen por variante - años {sc} (azar por estrato = 10%)\n",
                   to_markdown(resumen[resumen["ALCANCE"] == sc][cols])]
        md.append("\n## Diferencias pareadas con IC 95% (bootstrap por municipio)\n")
        md.append(
            "Formato: diferencia [IC 95%]; * = el intervalo excluye 0. Positivo = mejor, salvo DEVIANZA (negativo = mejor). "
            "TOTAL_CON_Q1_x = % de todos los desertores capturados cuando la lista aún atrapa x% de los desertores de Q1.\n"
        )
        for sc in resumen["ALCANCE"].unique():
            md += [f"\n### Años {sc}\n", to_markdown(diff_pivot(boot, sc))]
        md += [
            "\n## Calibración por año de prueba (esperados / observados; ideal = 1)\n", to_markdown(cal_piv),
            "\n## Calibración tras reescalar con el factor del año de prueba anterior\n", to_markdown(rec_piv),
            "\n## Devianza por año\n", to_markdown(dev_piv, nd=3),
            "\n## Devianza por año tras el reescalado\n", to_markdown(dev_rec, nd=3),
        ]
        (AUDIT_DIR / f"{stem}.md").write_text("\n".join(md), encoding="utf-8")
        plot_frontier(front, nombre)
    except OSError as err:
        print(f"[WARN] Could not write the report of {nombre}: {err}")


def run_panel(nombre: str, base_features: list) -> tuple:
    """Rolling-origin backtests: history-only GLM and offset GBM (B2 design) for every weighting and capacity."""
    pop = population(load_panel(nombre))
    feats = base_features + [COL_HIST_ADJ] + COLS_HIST_COUNTS
    rng = np.random.default_rng(SEED)
    store, params = {}, []
    obs = pop[pop[COL_ANIO] >= min(TEST_YEARS) - 1].groupby(COL_ANIO).agg(
        SEDES=("_d", "size"), DESERTORES=("_d", "sum"), ESTUDIANTES=(COL_PESO, "sum")
    )
    obs["TASA_OBSERVADA"] = obs["DESERTORES"] / obs["ESTUDIANTES"]
    observed = obs[["SEDES", "TASA_OBSERVADA"]].reset_index().rename(columns={COL_ANIO: "ANIO"})
    for anio in TEST_YEARS:
        train, test = pop[pop[COL_ANIO] < anio], pop[pop[COL_ANIO] == anio]
        if train.empty or test.empty:
            print(f"[WARN] {nombre} {anio}: no train or test rows, skipped.")
            continue
        k_eb = estimate_k_empirical_bayes(train["_d"], train[COL_PESO])
        if not np.isfinite(k_eb):
            raise ValueError(f"Empirical-Bayes K could not be estimated for {nombre} {anio}.")
        train, test = add_shrunk_history(train, test, k_eb)
        n_pred = test[COL_SIZE].fillna(test[COL_SIZE].median()).clip(lower=1).to_numpy()
        store[anio] = dict(
            d=test["_d"].to_numpy(), peso=test[COL_PESO].to_numpy(), n_pred=n_pred,
            quintil=pd.qcut(pd.Series(n_pred).rank(method="first"), 5, labels=False).to_numpy(),
            jitter=rng.random(len(test)) * 1e-12, mpio=test[COL_MPIO].astype(str).to_numpy(), preds={},
        )
        fold = {"ANIO_TEST": anio, "FILAS_TRAIN": len(train), "FILAS_TEST": len(test), "K_EB": k_eb}
        for w in WEIGHT_SCHEMES:
            w_train = make_weights(train, w, k_eb)
            glm = fit_history_glm(train, w_train)
            off_train, off_test = glm_linear_predictor(glm, train), glm_linear_predictor(glm, test)
            fold[f"INTERCEPTO_w{w}"], fold[f"PENDIENTE_w{w}"] = glm.intercept_, float(glm.coef_[0])
            store[anio]["preds"][f"GLM_solo_w{w}"] = np.exp(off_test)
            for cap_key, cap in CAPACITIES.items():
                store[anio]["preds"][f"B2_w{w}_{cap_key}"] = fit_predict_lgbm(train, test, feats, cap, w_train, off_train, off_test)
        params.append(fold)
        print(f"{nombre} | test {anio} | K = {k_eb:.1f} | train {len(train):,} | test {len(test):,} | done")
    pairs = [(f"B2_wsat_{c}", f"B2_wn_{c}") for c in CAPACITIES]                       # effect of the saturated weights
    pairs += [("GLM_solo_wsat", "GLM_solo_wn")]
    pairs += [(f"B2_w{w}_{c}", f"B2_w{w}_c3") for w in WEIGHT_SCHEMES for c in ("c1", "c2")]  # effect of less capacity
    pairs += [(f"B2_w{w}_{c}", f"GLM_solo_w{w}") for w in WEIGHT_SCHEMES for c in CAPACITIES]  # added value of the GBM
    return store, pairs, pd.DataFrame(params), observed


# =============================================================================
# 3. EXECUTION
# =============================================================================

def main() -> None:
    for nombre, features in PANELES.items():
        store, pairs, params, observed = run_panel(nombre, features)
        boot, resumen = bootstrap_pairs(store, pairs, N_BOOT, nombre)
        variants = sorted({v for pair in pairs for v in pair})
        front = frontier_table(store, variants, nombre)
        calib = calibration_table(store, variants, nombre)
        save_report(nombre, boot, resumen, front, calib, params, observed)
        print(f"\n=== {nombre}: GLM / K per fold ===")
        print(params.round(3).to_string(index=False))
        for sc in resumen["ALCANCE"].unique():
            print(f"\n=== {nombre}: summary per variant, years {sc} (random = 10% per stratum) ===")
            print(resumen[resumen["ALCANCE"] == sc].drop(columns="ALCANCE").round(2).to_string(index=False))
        print(f"\n=== {nombre}: paired differences, all years (* = 95% interval excludes 0) ===")
        print(diff_pivot(boot, "TODOS").to_string(index=False))
        print(f"\n=== {nombre}: calibration by year (expected / observed) ===")
        print(calib.pivot_table(index="CONJUNTO", columns="ANIO_TEST", values="CALIBRACION").round(3).to_string())


if __name__ == "__main__":
    main()


In [ ]:
"""
=============================================================================
PRE-MODELING 6: SHAP by families on the chosen model (B2, saturated weights, capacity c2)
=============================================================================
Model (decision of experiments 4 and 5): Poisson GLM on log(shrunk history) used as OFFSET + LightGBM Poisson that also
sees the history features, fitted with saturated weights n*K/(n+K) and capacity c2 (100 trees, min_child_samples 150).

What is explained: log(rate) = offset (history GLM) + sum of the TreeSHAP contributions of the GBM + a constant.
Each sede gets one NET contribution per family (sum of its features' SHAP values), in log-rate units; the family
"Historia propia" also carries the offset (centered) and is reported apart. The "without history" shares renormalise
the remaining families, which is the explanation that matters for policy. A contribution c means a relative risk of
exp(c) - 1 against an average sede (e.g. +0.10 -> +10.5%).

Blocks of the report (one md per panel + one figure; no csv):
  1. Sanity: additivity of SHAP (must be ~0) and the alpha criterion (capture of Q1 and of the total at alpha 0.20 / 0.25).
  2. Family shares by test year (stability) and 95% bootstrap interval (by municipality) for the last year.
  3. Direction and size of each variable (top vs bottom quartile, or 1 vs 0 for flags) in the last year.
  4. Objective 5: inside the top-10% priority list at ALPHA_POLICY, which non-history family is the main upward driver.

Limits: path-dependent TreeSHAP shares credit among correlated variables, so a family's share is an association inside
the model, not a causal effect; the direction table inherits the same caveat.

Input : Data/Processed/panel_final_<panel>.csv
Output: Data/Auditorio_Panel_Final/pre_modelo_shap_familias_<panel>.md and pre_modelo_shap_familias.png
=============================================================================
"""

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import PoissonRegressor

try:
    import lightgbm as lgb
except ImportError as err:  # corporate machines sometimes block pip: fail early with a clear message
    raise ImportError("lightgbm is required for this experiment (pip install lightgbm).") from err

# =============================================================================
# 0. PATHS
# =============================================================================
BASE_DIR = Path(
    r"C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data"
)
IN_DIR = BASE_DIR / "Processed"
AUDIT_DIR = BASE_DIR / "Auditorio_Panel_Final"

# =============================================================================
# 1. GENERAL VARIABLES
# =============================================================================
COL_SEDE, COL_ANIO, COL_MPIO = "SEDE_CODIGO", "ANIO_OBJETIVO", "COD_MPIO_DANE"
COL_RATE, COL_PESO, COL_SIZE = "TASA_DESERCION_SEDE", "PESO_ESTUDIANTES", "MATRICULA_TOTAL"
COL_HIST = "DESERCION_LAG3"
COL_HIST_ADJ = "DESERCION_LAG3_AJUSTADA"
COLS_HIST_COUNTS = ["ESTUDIANTES_LAG3", "DESERTORES_LAG3"]
HIST_LAG = 3
FAM_HIST = "Historia propia"

PREDICTORES_COMUNES = [
    "DESERCION_LAG3", "MATRICULA_TOTAL", "MATRICULA_PCT_CAMBIO", "PROP_DESPLAZADOS",
    "PROP_LIMITACION", "PROP_ETNIA", "PROP_FLEXIBLE", "FLAG_PDET", "FLAG_ZOMAC",
]
PREDICTORES_ICFES = ["ICFES_PROM_PUNT_GLOBAL", "ICFES_PROM_INSE"]
PANELES = {
    "ofrecen11": PREDICTORES_COMUNES + PREDICTORES_ICFES,
    "Solo_Primaria": PREDICTORES_COMUNES,
}
FAMILIAS = {
    FAM_HIST: [COL_HIST, COL_HIST_ADJ] + COLS_HIST_COUNTS,
    "Tamaño y dinámica de matrícula": ["MATRICULA_TOTAL", "MATRICULA_PCT_CAMBIO"],
    "Composición del alumnado": ["PROP_DESPLAZADOS", "PROP_LIMITACION", "PROP_ETNIA", "PROP_FLEXIBLE"],
    "Contexto territorial": ["FLAG_PDET", "FLAG_ZOMAC"],
    "ICFES": ["ICFES_PROM_PUNT_GLOBAL", "ICFES_PROM_INSE"],
}

TEST_YEARS = [2021, 2022, 2023]
TOP_SHARE = 0.10
ALPHA_POLICY = 0.20                  # priority = rate x n^alpha used to build the list explained in block 4
ALPHAS_CHECK = [0.20, 0.25]          # alpha criterion: Q1 capture must stay >= 10% (random list)
N_TREES, MIN_CHILD = 100, 150        # capacity c2
N_BOOT = 200
SEED = 42

LGBM_PARAMS = dict(
    objective="poisson", learning_rate=0.05, num_leaves=15, max_depth=4, reg_lambda=1.0,
    random_state=SEED, n_jobs=4, verbose=-1,
)

# =============================================================================
# 2. FUNCTION DEFINITIONS
# =============================================================================

def estimate_k_empirical_bayes(deserters, students) -> float:
    """Empirical-Bayes strength of the prior (K = alpha + beta of a Beta-Binomial), method of moments (Kleinman, 1973)."""
    d, n = np.asarray(deserters, float), np.asarray(students, float)
    ok = np.isfinite(d) & np.isfinite(n) & (n > 0)
    d, n = d[ok], n[ok]
    big_n = len(d)
    if big_n < 3 or n.sum() - big_n <= 0:
        return np.nan
    m = d.sum() / n.sum()
    if not 0 < m < 1:
        return np.nan
    s = np.sum(n * (d / n - m) ** 2)
    rho = float(np.clip((s / (m * (1 - m)) - (big_n - 1)) / (n.sum() - big_n), 1e-6, 0.99))
    return (1 - rho) / rho


def load_panel(nombre: str) -> pd.DataFrame:
    """Final panel plus the students and deserters of the same sede in t-3, joined by calendar year."""
    df = pd.read_csv(
        IN_DIR / f"panel_final_{nombre}.csv", dtype={COL_SEDE: str, COL_MPIO: str}, low_memory=False
    )
    for col in (COL_PESO, COL_MPIO):
        if col not in df.columns:
            raise KeyError(f"{col} is missing from panel_final_{nombre}.csv: rerun the master panel and Phase 2.")
    hist = df[[COL_SEDE, COL_ANIO, COL_RATE, COL_PESO]].copy()
    hist[COL_ANIO] = hist[COL_ANIO] + HIST_LAG
    hist = hist.rename(columns={COL_RATE: "_rate_lag3", COL_PESO: "ESTUDIANTES_LAG3"})
    df = df.merge(hist, on=[COL_SEDE, COL_ANIO], how="left", validate="many_to_one")
    df["DESERTORES_LAG3"] = np.round(df["_rate_lag3"] * df["ESTUDIANTES_LAG3"])  # NaN where there is no record
    both = df["_rate_lag3"].notna() & df[COL_HIST].notna()
    if both.any() and not np.allclose(df.loc[both, "_rate_lag3"], df.loc[both, COL_HIST], atol=1e-9):
        raise ValueError("The rate rebuilt for t-3 does not match DESERCION_LAG3: check the panel.")
    return df


def population(df: pd.DataFrame) -> pd.DataFrame:
    """Eligible rows with a Sit_Acad record (students known); deserters rebuilt from rate x students."""
    pop = df[(df["ELEGIBLE_MODELO"] == 1) & df[COL_PESO].gt(0) & df[COL_RATE].notna()].copy()
    pop["_d"] = np.round(pop[COL_RATE] * pop[COL_PESO])
    return pop


def add_shrunk_history(train: pd.DataFrame, test: pd.DataFrame, k: float) -> tuple:
    """DESERCION_LAG3_AJUSTADA = (deserters_t-3 + K*mean) / (students_t-3 + K); K and mean from TRAIN only, always > 0."""
    m_all = train["_d"].sum() / train[COL_PESO].sum()
    out = []
    for frame in (train, test):
        frame = frame.copy()
        known = frame["ESTUDIANTES_LAG3"].gt(0)
        shrunk = (frame["DESERTORES_LAG3"] + k * m_all) / (frame["ESTUDIANTES_LAG3"] + k)
        frame[COL_HIST_ADJ] = np.where(known, shrunk, m_all)
        out.append(frame)
    return out[0], out[1]


def fit_b2(train: pd.DataFrame, test: pd.DataFrame, feats: list, k: float) -> tuple:
    """History GLM (offset) + Poisson LightGBM with saturated weights; returns the model, the test offset and the raw GBM score."""
    n = train[COL_PESO].to_numpy(float)
    weights = n * k / (n + k)
    glm = PoissonRegressor(alpha=0.0, max_iter=500)
    glm.fit(np.log(train[[COL_HIST_ADJ]].to_numpy()), train[COL_RATE].to_numpy(), sample_weight=weights)
    off_train = glm.intercept_ + glm.coef_[0] * np.log(train[COL_HIST_ADJ].to_numpy())
    off_test = glm.intercept_ + glm.coef_[0] * np.log(test[COL_HIST_ADJ].to_numpy())
    mdl = lgb.LGBMRegressor(**LGBM_PARAMS, n_estimators=N_TREES, min_child_samples=MIN_CHILD)
    mdl.fit(train[feats], train[COL_RATE].to_numpy(), sample_weight=weights, init_score=off_train)
    return mdl, off_test, mdl.predict(test[feats], raw_score=True)


def shap_contributions(mdl, test: pd.DataFrame, feats: list, raw: np.ndarray) -> np.ndarray:
    """TreeSHAP contributions per variable (log scale); checks additivity against the raw score."""
    contrib = mdl.predict(test[feats], pred_contrib=True)
    bias, contrib = contrib[:, -1], contrib[:, :-1]
    err = float(np.abs(contrib.sum(axis=1) + bias - raw).max())
    if err > 1e-6:
        raise ValueError(f"SHAP additivity check failed (max error {err:.2e}).")
    return contrib


def family_contributions(contrib: np.ndarray, feats: list, off_test: np.ndarray) -> pd.DataFrame:
    """Net contribution of each family per sede (log scale); the history family also carries the centered offset."""
    fam = {}
    for name, cols in FAMILIAS.items():
        idx = [feats.index(c) for c in cols if c in feats]
        if idx:
            fam[name] = contrib[:, idx].sum(axis=1)
    fam[FAM_HIST] = fam[FAM_HIST] + (off_test - off_test.mean())
    return pd.DataFrame(fam)


def family_shares(fam: pd.DataFrame) -> pd.DataFrame:
    """Share of the mean absolute net contribution: with all families and without the history family."""
    m = fam.abs().mean()
    no_hist = m.drop(FAM_HIST)
    return pd.DataFrame({
        "PCT_TOTAL": 100 * m / m.sum(),
        "PCT_SIN_HISTORIA": 100 * no_hist / no_hist.sum(),
    })


def bootstrap_shares(fam: pd.DataFrame, mpio: np.ndarray, n_boot: int) -> pd.DataFrame:
    """95% percentile interval of the family shares, resampling municipalities."""
    codes, uniques = pd.factorize(mpio)
    abs_mat = fam.abs().to_numpy()
    cols = list(fam.columns)
    h = cols.index(FAM_HIST)
    rng = np.random.default_rng(SEED)
    tot, sin = [], []
    for _ in range(n_boot):
        w = np.bincount(rng.integers(0, len(uniques), len(uniques)), minlength=len(uniques))[codes]
        m = (abs_mat * w[:, None]).sum(axis=0) / w.sum()
        tot.append(100 * m / m.sum())
        no_h = np.delete(m, h)
        sin.append(100 * m / no_h.sum())
    tot, sin = np.array(tot), np.array(sin)
    sin_lo, sin_hi = np.percentile(sin, 2.5, axis=0), np.percentile(sin, 97.5, axis=0)
    sin_lo[h] = sin_hi[h] = np.nan       # the history family has no "without history" share
    return pd.DataFrame({
        "TOTAL_IC95_BAJO": np.percentile(tot, 2.5, axis=0), "TOTAL_IC95_ALTO": np.percentile(tot, 97.5, axis=0),
        "SIN_HIST_IC95_BAJO": sin_lo, "SIN_HIST_IC95_ALTO": sin_hi,
    }, index=cols)


def feature_effects(contrib: np.ndarray, test: pd.DataFrame, feats: list) -> pd.DataFrame:
    """Per variable: mean |SHAP|, Spearman(value, SHAP) and the % change in risk between high and low values (quartiles; 1 vs 0 for flags)."""
    fam_of = {c: f for f, cols in FAMILIAS.items() for c in cols}
    filas = []
    for j, c in enumerate(feats):
        if c in FAMILIAS[FAM_HIST]:   # history = offset + its own family row; its GBM-only SHAP alone would mislead
            continue
        x, s = test[c].to_numpy(float), contrib[:, j]
        ok = np.isfinite(x)
        if ok.sum() < 50 or np.unique(x[ok]).size < 2:
            continue
        uniq = np.unique(x[ok])
        if uniq.size <= 2:                                  # flags: value 1 vs value 0
            lo, hi = ok & (x == uniq[0]), ok & (x == uniq[-1])
            groups_differ = True
        else:                                               # continuous: top vs bottom quartile
            q25, q75 = np.percentile(x[ok], [25, 75])
            lo, hi = ok & (x <= q25), ok & (x >= q75)
            groups_differ = q25 < q75
        diff = s[hi].mean() - s[lo].mean() if (groups_differ and lo.any() and hi.any()) else np.nan
        filas.append({
            "FAMILIA": fam_of.get(c, "Otra"), "VARIABLE": c, "MEAN_ABS_SHAP": float(np.abs(s).mean()),
            "SPEARMAN_VALOR_SHAP": float(spearmanr(x[ok], s[ok])[0]), "EFECTO_ALTO_VS_BAJO_PCT": 100 * (np.exp(diff) - 1),
        })
    return pd.DataFrame(filas).sort_values("MEAN_ABS_SHAP", ascending=False)


def list_capture(d, rate_hat, n_pred, quintil, alpha) -> tuple:
    """Top-10% list by rate x n^alpha: returns (mask, % of Q1 deserters caught, % of all deserters caught)."""
    score = rate_hat * n_pred ** alpha
    top = np.zeros(len(d), bool)
    top[np.argsort(-score)[: int(np.ceil(TOP_SHARE * len(d)))]] = True
    return top, 100 * d[top & (quintil == 0)].sum() / d[quintil == 0].sum(), 100 * d[top].sum() / d.sum()


def priority_list_drivers(fam: pd.DataFrame, top: np.ndarray) -> pd.DataFrame:
    """Objective 5: share of sedes (inside the list and outside it) whose main upward non-history driver is each family."""
    non_hist = [c for c in fam.columns if c != FAM_HIST]
    arr = fam[non_hist].to_numpy()
    dominant = np.where(arr.max(axis=1) > 0, np.array(non_hist)[arr.argmax(axis=1)], "Ninguna (todas reducen)")
    out = []
    for name in non_hist + ["Ninguna (todas reducen)"]:
        row = {"FAMILIA_DOMINANTE": name,
               "PCT_SEDES_LISTA": 100 * float((dominant[top] == name).mean()),
               "PCT_SEDES_RESTO": 100 * float((dominant[~top] == name).mean())}
        if name in fam.columns:
            row["EFECTO_MEDIO_LISTA_PCT"] = 100 * (np.exp(fam.loc[top, name].mean()) - 1)
            row["EFECTO_MEDIO_RESTO_PCT"] = 100 * (np.exp(fam.loc[~top, name].mean()) - 1)
        out.append(row)
    hist = {"FAMILIA_DOMINANTE": f"({FAM_HIST}, solo efecto medio)", "PCT_SEDES_LISTA": np.nan, "PCT_SEDES_RESTO": np.nan,
            "EFECTO_MEDIO_LISTA_PCT": 100 * (np.exp(fam.loc[top, FAM_HIST].mean()) - 1),
            "EFECTO_MEDIO_RESTO_PCT": 100 * (np.exp(fam.loc[~top, FAM_HIST].mean()) - 1)}
    return pd.DataFrame(out + [hist])


def to_markdown(df: pd.DataFrame, nd: int = 2) -> str:
    """Minimal markdown table (no extra dependency)."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for r in df.itertuples(index=False):
        lines.append("| " + " | ".join(f"{v:.{nd}f}" if isinstance(v, float) else str(v) for v in r) + " |")
    return "\n".join(lines)


def run_panel(nombre: str, base_features: list) -> dict:
    """Rolling-origin fits (train < year, explain year): SHAP families per year; full detail kept for the last year."""
    pop = population(load_panel(nombre))
    feats = base_features + [COL_HIST_ADJ] + COLS_HIST_COUNTS
    res = {"shares_year": [], "checks": []}
    for anio in TEST_YEARS:
        train, test = pop[pop[COL_ANIO] < anio], pop[pop[COL_ANIO] == anio]
        k_eb = estimate_k_empirical_bayes(train["_d"], train[COL_PESO])
        if not np.isfinite(k_eb):
            raise ValueError(f"Empirical-Bayes K could not be estimated for {nombre} {anio}.")
        train, test = add_shrunk_history(train, test, k_eb)
        mdl, off_test, raw = fit_b2(train, test, feats, k_eb)
        contrib = shap_contributions(mdl, test, feats, raw)
        fam = family_contributions(contrib, feats, off_test)
        shares = family_shares(fam)
        res["shares_year"].append(shares.assign(ANIO_TEST=anio).rename_axis("FAMILIA").reset_index())
        rate_hat = np.exp(raw + off_test)
        d = test["_d"].to_numpy()
        n_pred = test[COL_SIZE].fillna(test[COL_SIZE].median()).clip(lower=1).to_numpy()
        quintil = pd.qcut(pd.Series(n_pred).rank(method="first"), 5, labels=False).to_numpy()
        for a in ALPHAS_CHECK:
            _, q1, tot = list_capture(d, rate_hat, n_pred, quintil, a)
            res["checks"].append({"ANIO_TEST": anio, "ALPHA": a, "CAPTURA_Q1": q1, "CAPTURA_TOTAL": tot})
        print(f"{nombre} | test {anio} | K = {k_eb:.1f} | train {len(train):,} | test {len(test):,} | SHAP ok")
        if anio == TEST_YEARS[-1]:
            top, _, _ = list_capture(d, rate_hat, n_pred, quintil, ALPHA_POLICY)
            res.update(
                effects=feature_effects(contrib, test, feats),
                boot=bootstrap_shares(fam, test[COL_MPIO].astype(str).to_numpy(), N_BOOT),
                drivers=priority_list_drivers(fam, top), shares_last=shares,
            )
    res["shares_year"] = pd.concat(res["shares_year"])
    res["checks"] = pd.DataFrame(res["checks"])
    return res


def save_report(nombre: str, res: dict) -> None:
    """Write the md report of one panel; a locked OneDrive file must not abort the experiment."""
    try:
        AUDIT_DIR.mkdir(parents=True, exist_ok=True)
        chk = res["checks"]
        chk_mean = chk.groupby("ALPHA", as_index=False)[["CAPTURA_Q1", "CAPTURA_TOTAL"]].mean().assign(ANIO_TEST="Promedio")
        chk_all = pd.concat([chk, chk_mean[chk.columns]], ignore_index=True)
        chk_all["x_AZAR_Q1"] = chk_all["CAPTURA_Q1"] / (100 * TOP_SHARE)
        by_year = res["shares_year"].pivot_table(index="FAMILIA", columns="ANIO_TEST", values=["PCT_TOTAL", "PCT_SIN_HISTORIA"])
        by_year.columns = [f"{a}_{b}" for a, b in by_year.columns]
        last = res["shares_last"].join(res["boot"]).rename_axis("FAMILIA").reset_index()
        md = [
            f"# SHAP por familias - panel {nombre} (B2, pesos saturados, c2)",
            "\n## 1. Criterio de alpha: captura de la lista del 10% (azar de Q1 = 10%)\n",
            to_markdown(chk_all, nd=2),
            "\n## 2. Participación por familia y año (contribución neta absoluta media, %)\n",
            to_markdown(by_year.reset_index(), nd=1),
            f"\n### Último año ({TEST_YEARS[-1]}) con IC 95% (bootstrap por municipio, {N_BOOT} réplicas)\n",
            to_markdown(last, nd=1),
            f"\n## 3. Dirección y tamaño por variable ({TEST_YEARS[-1]})\n",
            "EFECTO_ALTO_VS_BAJO_PCT = cambio de riesgo relativo entre el cuartil alto y el bajo de la variable (1 vs 0 en banderas); "
            "es asociación dentro del modelo, no causalidad. La historia no se desglosa: vive en el offset y en su propia familia.\n",
            to_markdown(res["effects"], nd=3),
            f"\n## 4. Lista prioritaria (top {int(TOP_SHARE * 100)}% con alpha = {ALPHA_POLICY}): familia que más sube el riesgo sin contar la historia\n",
            to_markdown(res["drivers"], nd=1),
        ]
        (AUDIT_DIR / f"pre_modelo_shap_familias_{nombre}.md").write_text("\n".join(md), encoding="utf-8")
    except OSError as err:
        print(f"[WARN] Could not write the report of {nombre}: {err}")


def plot_shares(results: dict) -> None:
    """One figure, one subplot per panel: share of each family with and without the history; shown inline and saved as png."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        return
    fig, axes = plt.subplots(1, len(results), figsize=(6.5 * len(results), 4.5), squeeze=False)
    for ax, (nombre, res) in zip(axes[0], results.items()):
        sh = res["shares_last"].sort_values("PCT_TOTAL")
        pos = np.arange(len(sh))
        ax.barh(pos + 0.2, sh["PCT_TOTAL"], height=0.4, label="With history")
        ax.barh(pos - 0.2, sh["PCT_SIN_HISTORIA"].fillna(0), height=0.4, label="Without history")
        ax.set_yticks(pos)
        ax.set_yticklabels(sh.index, fontsize=8)
        ax.set_xlabel("% of the mean absolute net contribution")
        ax.set_title(f"{nombre} ({TEST_YEARS[-1]})")
        ax.legend(fontsize=8)
    fig.tight_layout()
    try:
        AUDIT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(AUDIT_DIR / "pre_modelo_shap_familias.png", dpi=150, bbox_inches="tight")
    except OSError as err:
        print(f"[WARN] Could not save the figure: {err}")
    plt.show()


# =============================================================================
# 3. EXECUTION
# =============================================================================

def main() -> None:
    results = {}
    for nombre, features in PANELES.items():
        results[nombre] = run_panel(nombre, features)
        save_report(nombre, results[nombre])
        res = results[nombre]
        print(f"\n=== {nombre}: alpha criterion (Q1 capture vs 10% random) ===")
        print(res["checks"].round(2).to_string(index=False))
        print(f"\n=== {nombre}: family shares by year (%) ===")
        print(res["shares_year"].pivot_table(index="FAMILIA", columns="ANIO_TEST", values=["PCT_TOTAL", "PCT_SIN_HISTORIA"]).round(1).to_string())
        print(f"\n=== {nombre}: {TEST_YEARS[-1]} shares with 95% interval ===")
        print(res["shares_last"].join(res["boot"]).round(1).to_string())
        print(f"\n=== {nombre}: variable effects {TEST_YEARS[-1]} ===")
        print(res["effects"].round(3).to_string(index=False))
        print(f"\n=== {nombre}: priority list drivers (alpha = {ALPHA_POLICY}) ===")
        print(res["drivers"].round(1).to_string(index=False))
    plot_shares(results)


if __name__ == "__main__":
    main()


In [ ]:
"""
=============================================================================
PRE-MODELING 7: the two classification targets (level class and variation class) and their noise
=============================================================================
Decisions already taken with the advisor: multiclass classification; dropout level in at most 4 classes with the ZERO as its
own class ("sin deserción observada") and static cut-offs (percentiles of ALL the available positive rates, frozen for
every year); a second target for the variation (baja / igual / sube) with respect to the last known rate (t-3);
lags 3, 4 and 5 allowed (only t-3 is certain at scoring time, since sede-level data end 3 years before the target).

What this cell does (no model is fitted):
  1. Lag value: how many rows have lag 4 / lag 5 and whether they add rank information over lag 3 (Spearman).
  2. Rate distribution: share of zeros and percentiles of the positive rate by size stratum.
  3. LEVEL candidates: 2 rate definitions (observed / empirical-Bayes adjusted) x 3 percentile schemes. For each one:
     numeric cut-offs, class shares by year (stability of a static cut-off), size mix of the "alto" class, and PERSISTENCE
     (class at t-3 as the prediction of the class at t: accuracy, student-weighted accuracy, quadratic kappa, by stratum).
     That persistence is the baseline every classifier must beat.
  4. VARIATION candidates: fixed percentage points (0.1 = advisor's example), relative change on the adjusted rate, and
     funnel-plot limits on the binomial standard error of the difference (95% = 1.96, 99.8% = 3.09; Spiegelhalter, 2005), the
     last two also with a robust overdispersion factor. For each rule: shares of baja/igual/sube and a NOISE DIAGNOSTIC:
     how many changes the rule would flag if NO sede had changed at all (binomial noise only), overall and in Q1/Q5.
  5. Regression to the mean: variation of each sede according to its level class at t-3.

Literature used for the choices: small units sit disproportionately at the extremes and year-to-year changes are mostly
sampling noise (Kane & Staiger, 2002; empirical-Bayes filtering); funnel-plot control limits at 2 and 3 SD
(Spiegelhalter, 2005); overdispersion adjustment (Spiegelhalter, 2005/2012).

Input : Data/Processed/panel_final_<panel>.csv
Output: Data/Auditorio_Panel_Final/pre_modelo_objetivos_clasificacion_<panel>.md
=============================================================================
"""

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score

# =============================================================================
# 0. PATHS
# =============================================================================
BASE_DIR = Path(
    r"C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data"
)
IN_DIR = BASE_DIR / "Processed"
AUDIT_DIR = BASE_DIR / "Auditorio_Panel_Final"

# =============================================================================
# 1. GENERAL VARIABLES
# =============================================================================
COL_SEDE, COL_ANIO = "SEDE_CODIGO", "ANIO_OBJETIVO"
COL_RATE, COL_PESO, COL_SIZE = "TASA_DESERCION_SEDE", "PESO_ESTUDIANTES", "MATRICULA_TOTAL"
COL_HIST = "DESERCION_LAG3"
LAGS = [3, 4, 5]
PANELES = ["ofrecen11", "Solo_Primaria"]

LEVEL_LABELS = {0: "0 Sin deserción observada", 1: "1 Bajo", 2: "2 Medio", 3: "3 Alto"}
ALTO = 3
RATE_DEFS = ["obs", "adj"]                         # observed rate / empirical-Bayes adjusted rate
LEVEL_SCHEMES = {                                  # percentiles of the POSITIVE rates (class 0 is the zero)
    "P50_P90": [50, 90],
    "TERCILES": [100 / 3, 200 / 3],
    "P75_P95": [75, 95],
}
VAR_LABELS = {0: "Baja", 1: "Igual", 2: "Sube"}
VAR_RULES = {                                      # (kind, parameter)
    "ABS_0.1pp": ("abs_pp", 0.1),
    "ABS_0.5pp": ("abs_pp", 0.5),
    "ABS_1pp": ("abs_pp", 1.0),
    "REL_10pct": ("rel_adj", 10),
    "REL_25pct": ("rel_adj", 25),
    "FUNNEL_95": ("stat", 1.96),
    "FUNNEL_99.8": ("stat", 3.09),
    "FUNNEL_OD_95": ("stat_od", 1.96),
    "FUNNEL_OD_99.8": ("stat_od", 3.09),
}
DIAG_DEF, DIAG_SCHEME, DIAG_RULE = "adj", "P50_P90", "FUNNEL_95"   # used for the regression-to-the-mean table
N_SIM = 5                                          # null simulations per row
SEED = 42

# =============================================================================
# 2. FUNCTION DEFINITIONS
# =============================================================================

def estimate_k_empirical_bayes(deserters, students) -> float:
    """Empirical-Bayes strength of the prior (K = alpha + beta of a Beta-Binomial), method of moments (Kleinman, 1973)."""
    d, n = np.asarray(deserters, float), np.asarray(students, float)
    ok = np.isfinite(d) & np.isfinite(n) & (n > 0)
    d, n = d[ok], n[ok]
    big_n = len(d)
    m = d.sum() / n.sum()
    s = np.sum(n * (d / n - m) ** 2)
    rho = float(np.clip((s / (m * (1 - m)) - (big_n - 1)) / (n.sum() - big_n), 1e-6, 0.99))
    return (1 - rho) / rho


def load_panel_with_lags(nombre: str) -> pd.DataFrame:
    """Final panel plus deserters and students of the same sede in t-3, t-4 and t-5, joined by calendar year."""
    df = pd.read_csv(IN_DIR / f"panel_final_{nombre}.csv", dtype={COL_SEDE: str}, low_memory=False)
    if COL_PESO not in df.columns:
        raise KeyError(f"{COL_PESO} is missing from panel_final_{nombre}.csv: rerun the master panel and Phase 2.")
    for k in LAGS:
        hist = df[[COL_SEDE, COL_ANIO, COL_RATE, COL_PESO]].copy()
        hist[COL_ANIO] = hist[COL_ANIO] + k
        hist = hist.rename(columns={COL_RATE: f"_rate_lag{k}", COL_PESO: f"ESTUDIANTES_LAG{k}"})
        df = df.merge(hist, on=[COL_SEDE, COL_ANIO], how="left", validate="many_to_one")
        df[f"DESERTORES_LAG{k}"] = np.round(df[f"_rate_lag{k}"] * df[f"ESTUDIANTES_LAG{k}"])
    both = df["_rate_lag3"].notna() & df[COL_HIST].notna()
    if both.any() and not np.allclose(df.loc[both, "_rate_lag3"], df.loc[both, COL_HIST], atol=1e-9):
        raise ValueError("The rate rebuilt for t-3 does not match DESERCION_LAG3: check the panel.")
    return df


def adjust_rate(d, n, k, m):
    """Empirical-Bayes adjusted rate (d + K*m) / (n + K): always > 0 and pulled towards the mean for small sedes."""
    return (d + k * m) / (n + k)


def rate_of(d, n, definition, k, m):
    """Observed rate or adjusted rate."""
    return d / n if definition == "obs" else adjust_rate(d, n, k, m)


def level_cuts(all_d, all_n, definition, percentiles, k, m) -> np.ndarray:
    """Static cut-offs: percentiles of the positive rates of ALL the available sede-years."""
    pos = all_d > 0
    return np.percentile(rate_of(all_d[pos], all_n[pos], definition, k, m), percentiles)


def level_class(d, n, definition, cuts, k, m) -> np.ndarray:
    """0 = no observed dropout; 1..3 = position of the rate among the cut-offs; -1 = no record."""
    valid = np.isfinite(d) & np.isfinite(n) & (n > 0)
    out = np.full(len(d), -1)
    dv, nv = d[valid], n[valid]
    out[valid] = np.where(dv > 0, 1 + np.searchsorted(cuts, rate_of(dv, nv, definition, k, m), side="right"), 0)
    return out


def persistence_stats(c_t, c_3, weights, strata) -> dict:
    """The class at t-3 used as prediction of the class at t: accuracy, weighted accuracy, quadratic kappa, 'alto' persistence."""
    ok = c_3 >= 0
    hit = (c_t == c_3)
    out = {
        "ACC": 100 * hit[ok].mean(),
        "ACC_POND": 100 * np.average(hit[ok], weights=weights[ok]),
        "KAPPA_CUAD": cohen_kappa_score(c_3[ok], c_t[ok], weights="quadratic"),
        "ALTO_PERSISTE": 100 * ((c_t == ALTO) & (c_3 == ALTO) & ok).sum() / max(((c_3 == ALTO) & ok).sum(), 1),
        "ALTO_VENIA_ALTO": 100 * ((c_t == ALTO) & (c_3 == ALTO) & ok).sum() / max(((c_t == ALTO) & ok).sum(), 1),
    }
    for q in (0, 4):
        sel = ok & (strata == q)
        out[f"KAPPA_Q{q + 1}"] = cohen_kappa_score(c_3[sel], c_t[sel], weights="quadratic") if sel.sum() > 50 else np.nan
    return out


def variation_class(rule, d_t, n_t, d_3, n_3, k, m, phi) -> np.ndarray:
    """0 = baja, 1 = igual, 2 = sube, comparing the rate at t with the rate at t-3 under the chosen rule."""
    kind, param = rule
    r_t, r_3 = d_t / n_t, d_3 / n_3
    if kind == "abs_pp":
        delta, thr = r_t - r_3, param / 100
    elif kind == "rel_adj":
        delta, thr = adjust_rate(d_t, n_t, k, m) / adjust_rate(d_3, n_3, k, m) - 1, param / 100
    else:   # funnel limits on the binomial standard error of the difference between the two years
        p0 = (d_t + d_3 + k * m) / (n_t + n_3 + k)
        se = np.sqrt(p0 * (1 - p0) * (1 / n_t + 1 / n_3))
        delta, thr = r_t - r_3, param * se * (np.sqrt(phi) if kind == "stat_od" else 1.0)
    return np.where(delta > thr, 2, np.where(delta < -thr, 0, 1))


def overdispersion_factor(d_t, n_t, d_3, n_3, k, m) -> float:
    """Robust (MAD based) multiplicative overdispersion of the standardized year-to-year difference; >= 1."""
    p0 = (d_t + d_3 + k * m) / (n_t + n_3 + k)
    z = (d_t / n_t - d_3 / n_3) / np.sqrt(p0 * (1 - p0) * (1 / n_t + 1 / n_3))
    return max(1.0, (1.4826 * np.median(np.abs(z - np.median(z)))) ** 2)


def to_markdown(df: pd.DataFrame, nd: int = 2) -> str:
    """Minimal markdown table (no extra dependency)."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for r in df.itertuples(index=False):
        lines.append("| " + " | ".join(f"{v:.{nd}f}" if isinstance(v, float) else str(v) for v in r) + " |")
    return "\n".join(lines)


def analyze_panel(nombre: str) -> list:
    """Build every table of the report for one panel; returns a list of (title, text) sections."""
    rng = np.random.default_rng(SEED)
    df = load_panel_with_lags(nombre)
    allrows = df[df[COL_PESO].gt(0) & df[COL_RATE].notna()]
    all_d, all_n = np.round(allrows[COL_RATE] * allrows[COL_PESO]).to_numpy(), allrows[COL_PESO].to_numpy(float)
    m = all_d.sum() / all_n.sum()
    k = estimate_k_empirical_bayes(all_d, all_n)

    pop = df[(df["ELEGIBLE_MODELO"] == 1) & df[COL_PESO].gt(0) & df[COL_RATE].notna()].copy()
    pop["_d"] = np.round(pop[COL_RATE] * pop[COL_PESO])
    size = pop[COL_SIZE].fillna(pop[COL_SIZE].median())
    pop["_strat"] = pd.qcut(size.rank(method="first"), 5, labels=False)
    years = [int(y) for y in sorted(pop[COL_ANIO].unique())]
    sections = [("Base de cálculo", (
        f"Filas con registro (todos los años): {len(allrows):,}. Filas elegibles: {len(pop):,}. "
        f"K de Bayes empírico (todo lo disponible) = {k:.1f}; tasa media = {100 * m:.2f}%. Años objetivo: {years}."))]

    # ---- 1. value of lags 4 and 5 -------------------------------------------------------------
    rows = []
    for yr in years:
        sub = pop[pop[COL_ANIO] == yr]
        rows.append({"ANIO": yr, "FILAS": len(sub), **{f"PCT_LAG{kk}": 100 * sub[f"ESTUDIANTES_LAG{kk}"].gt(0).mean() for kk in LAGS}})
    sections.append(("1a. Disponibilidad de rezagos por año objetivo (% de filas con registro)", to_markdown(pd.DataFrame(rows), 1)))
    full = pop[(pop[[f"ESTUDIANTES_LAG{kk}" for kk in LAGS]] > 0).all(axis=1)]
    if len(full) > 200:
        adj = {kk: adjust_rate(full[f"DESERTORES_LAG{kk}"], full[f"ESTUDIANTES_LAG{kk}"], k, m) for kk in LAGS}
        pooled = adjust_rate(sum(full[f"DESERTORES_LAG{kk}"] for kk in LAGS), sum(full[f"ESTUDIANTES_LAG{kk}"] for kk in LAGS), k, m)
        cands = {"LAG3": adj[3], "LAG4": adj[4], "LAG5": adj[5], "LAG3+4+5 (agrupado)": pooled}
        rows = []
        for name, series in cands.items():
            row = {"PREDICTOR": name, "SPEARMAN_TODAS": spearmanr(series, full[COL_RATE])[0]}
            for q in (0, 4):
                sel = (full["_strat"] == q).to_numpy()
                row[f"SPEARMAN_Q{q + 1}"] = spearmanr(series[sel], full.loc[sel, COL_RATE])[0]
            rows.append(row)
        sections.append((f"1b. Valor de los rezagos: Spearman con la tasa observada en t (solo las {len(full):,} filas con los 3 rezagos)",
                         to_markdown(pd.DataFrame(rows), 3)))
    else:
        sections.append(("1b. Valor de los rezagos", "Muy pocas filas con los tres rezagos para comparar."))

    # ---- 2. distribution of the rate ---------------------------------------------------------
    rows = []
    for label, sel in [("Todas", pop["_strat"] >= 0)] + [(f"Q{q + 1}", pop["_strat"] == q) for q in range(5)]:
        sub = pop[sel]
        pos = sub.loc[sub["_d"] > 0, COL_RATE]
        rows.append({"ESTRATO": label, "SEDES": len(sub), "PCT_CERO": 100 * (sub["_d"] == 0).mean(),
                     **{f"P{p}": 100 * np.percentile(pos, p) for p in (25, 50, 75, 90, 95, 99)}})
    sections.append(("2. Distribución de la tasa observada (%) entre sedes con deserción positiva, por estrato de tamaño", to_markdown(pd.DataFrame(rows), 2)))

    # ---- 3. level candidates ---------------------------------------------------------------------
    d_t, n_t = pop["_d"].to_numpy(), pop[COL_PESO].to_numpy(float)
    d_3, n_3 = pop["DESERTORES_LAG3"].to_numpy(), pop["ESTUDIANTES_LAG3"].to_numpy(float)
    strata, weights = pop["_strat"].to_numpy(), n_t
    rows_cut, rows_year, rows_mix, rows_pers = [], [], [], []
    classes = {}
    for definition in RATE_DEFS:
        for scheme, pct in LEVEL_SCHEMES.items():
            cuts = level_cuts(all_d, all_n, definition, pct, k, m)
            c_t, c_3 = level_class(d_t, n_t, definition, cuts, k, m), level_class(d_3, n_3, definition, cuts, k, m)
            classes[(definition, scheme)] = (c_t, c_3)
            tag = f"{definition}/{scheme}"
            rows_cut.append({"DEFINICION/ESQUEMA": tag, "CORTE_1_PCT": 100 * cuts[0], "CORTE_2_PCT": 100 * cuts[1]})
            for cls, lab in LEVEL_LABELS.items():
                rows_year.append({"DEFINICION/ESQUEMA": tag, "CLASE": lab,
                                  **{str(yr): 100 * (c_t[(pop[COL_ANIO] == yr).to_numpy()] == cls).mean() for yr in years}})
            alto = c_t == ALTO
            rows_mix.append({"DEFINICION/ESQUEMA": tag, "SEDES_ALTO": int(alto.sum()),
                             **{f"PCT_ALTO_EN_Q{q + 1}": 100 * (strata[alto] == q).mean() for q in range(5)}})
            rows_pers.append({"DEFINICION/ESQUEMA": tag, **persistence_stats(c_t, c_3, weights, strata)})
    sections += [
        ("3a. Cortes estáticos de la tasa (%) entre sedes con deserción positiva (clase 0 = cero aparte)", to_markdown(pd.DataFrame(rows_cut), 3)),
        ("3b. Distribución de clases por año con cortes estáticos (% de sedes)", to_markdown(pd.DataFrame(rows_year), 1)),
        ("3c. Quién compone la clase Alto: % de sedes Alto por estrato de tamaño (Q1 = sedes más pequeñas)", to_markdown(pd.DataFrame(rows_mix), 1)),
        ("3d. Persistencia: la clase de t-3 como predicción de la clase de t (línea base a superar)", to_markdown(pd.DataFrame(rows_pers), 3)),
    ]

    # ---- 4. variation candidates and noise diagnostic ----------------------------------------------
    ok = (n_3 > 0) & (n_t > 0)
    phi = overdispersion_factor(d_t[ok], n_t[ok], d_3[ok], n_3[ok], k, m)
    p0 = (d_t[ok] + d_3[ok] + k * m) / (n_t[ok] + n_3[ok] + k)
    st = strata[ok]
    rows_var, var_obs = [], {}
    for name, rule in VAR_RULES.items():
        cls = variation_class(rule, d_t[ok], n_t[ok], d_3[ok], n_3[ok], k, m, phi)
        var_obs[name] = cls
        fake = []
        for _ in range(N_SIM):   # no real change: both years drawn from the same sede-level rate
            sd_t = rng.binomial(np.round(n_t[ok]).astype(int), p0)
            sd_3 = rng.binomial(np.round(n_3[ok]).astype(int), p0)
            fake.append(variation_class(rule, sd_t, n_t[ok], sd_3, n_3[ok], k, m, phi) != 1)
        fake = np.mean(fake, axis=0)
        changed = cls != 1
        rows_var.append({
            "REGLA": name, "PCT_BAJA": 100 * (cls == 0).mean(), "PCT_IGUAL": 100 * (cls == 1).mean(), "PCT_SUBE": 100 * (cls == 2).mean(),
            "CAMBIO_OBS_Q1": 100 * changed[st == 0].mean(), "CAMBIO_OBS_Q5": 100 * changed[st == 4].mean(),
            "CAMBIO_FALSO_TOTAL": 100 * fake.mean(), "CAMBIO_FALSO_Q1": 100 * fake[st == 0].mean(), "CAMBIO_FALSO_Q5": 100 * fake[st == 4].mean(),
            "RUIDO_SOBRE_CAMBIO_TOTAL": fake.mean() / max(changed.mean(), 1e-9),
        })
    sections += [
        (f"4. Reglas de variación (t vs t-3; {int(ok.sum()):,} sedes con ambos años; factor de sobredispersión robusto = {phi:.2f})",
         "CAMBIO_FALSO = % de sedes que la regla marcaría como Baja/Sube aunque NINGUNA hubiera cambiado (solo ruido binomial). "
         "RUIDO_SOBRE_CAMBIO_TOTAL = cambio falso / cambio observado (cerca de 1 = la regla detecta ruido).\n\n"
         + to_markdown(pd.DataFrame(rows_var), 2)),
    ]

    # ---- 5. regression to the mean -----------------------------------------------------------------
    c_t_d, c_3_d = classes[(DIAG_DEF, DIAG_SCHEME)]
    sel = (c_3_d[ok] >= 0)
    tab = pd.crosstab(pd.Series(c_3_d[ok][sel]).map(LEVEL_LABELS), pd.Series(var_obs[DIAG_RULE][sel]).map(VAR_LABELS), normalize="index") * 100
    sections.append((f"5. Regresión a la media: % de sedes Baja/Igual/Sube según su nivel en t-3 (nivel {DIAG_DEF}/{DIAG_SCHEME}, variación {DIAG_RULE})",
                     to_markdown(tab.rename_axis("NIVEL_T-3").reset_index(), 1)))
    return sections


def save_report(nombre: str, sections: list) -> None:
    """Write the md report of one panel; a locked OneDrive file must not abort the experiment."""
    try:
        AUDIT_DIR.mkdir(parents=True, exist_ok=True)
        md = [f"# Objetivos de clasificación (nivel y variación) - panel {nombre}"]
        for title, text in sections:
            md += [f"\n## {title}\n", text]
        (AUDIT_DIR / f"pre_modelo_objetivos_clasificacion_{nombre}.md").write_text("\n".join(md), encoding="utf-8")
    except OSError as err:
        print(f"[WARN] Could not write the report of {nombre}: {err}")


# =============================================================================
# 3. EXECUTION
# =============================================================================

def main() -> None:
    for nombre in PANELES:
        sections = analyze_panel(nombre)
        save_report(nombre, sections)
        print(f"\n################ {nombre} ################")
        for title, text in sections:
            print(f"\n=== {title} ===\n{text}")


if __name__ == "__main__":
    main()
